In [ ]:
!pip install vaderSentiment -q

import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack, csr_matrix
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

analyzer = SentimentIntensityAnalyzer()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)

In [ ]:
import pandas as pd

df = pd.read_csv('/content/final_combined_enriched_v4.csv')
print(f"Total tweets : {len(df):,}")
print(f"Columns      : {df.columns.tolist()}")
df.head(3)

Total tweets : 50,169
Columns      : ['tweet', 'type']


,tweet,type
0,This cartoon by Alok @caricatured speak about the reality of deals in Indian Politics - Backdoor - or Unholy nexus - when all guys unite on controlling money minting Cricket Administration 😎,1
1,I am going to post one #BernieAtTheFarmersProtest every day until either the Farmers protest ends or @BernieSanders @SenSanders notices this and highlights the issue. Or I get blocked by Twitter. #Berniememes #BernieSandersMittens #BernieSanders #FarmersProtest 34 https://t.co/2NFSWxE6fA,1
2,@sardesairajdeep @OfficialUrmila @RahulGandhi So bashing Islamic terrorism is communal but when someone talks about Hindu violence its crappy. Bloody hypocrites. 😪 \n#ModiLaoDeshBanao #ModiHainTohMumkinHain #LokSabhaElections2019 #CongressMuktBharat,1


In [ ]:
df = df.drop(columns=['type'])

if 'user' not in df.columns:
    df['user'] = ['user_' + str(i) for i in range(len(df))]

df_unlabeled = df[['user', 'tweet']].copy()

df_unlabeled = df_unlabeled.drop_duplicates(subset='tweet').reset_index(drop=True)
df_unlabeled = df_unlabeled[df_unlabeled['tweet'].str.strip() != ''].reset_index(drop=True)

print("Unlabeled shape :", df_unlabeled.shape)
print(df_unlabeled.head())


Unlabeled shape : (50034, 2)
     user  \
0  user_0   
1  user_1   
2  user_2   
3  user_3   
4  user_4   

                                                                                                                                                                                                                                                                                              tweet  
0                                                                                                   This cartoon by Alok @caricatured  speak about the reality of deals in Indian Politics - Backdoor - or Unholy nexus - when all guys unite on controlling money minting Cricket Administration 😎  
1  I am going to post one #BernieAtTheFarmersProtest every day until either the Farmers protest ends or @BernieSanders @SenSanders notices this and highlights the issue. Or I get blocked by Twitter. #Berniememes #BernieSandersMittens #BernieSanders #FarmersProtest 34 https://t.co/2NFSWxE6fA  
2         

In [ ]:

print(df.shape)


(50169, 2)


In [ ]:
df.isnull().sum()
df.duplicated().sum()

np.int64(0)

In [ ]:
import re

df['mention'] = df['tweet'].astype(str).apply(
    lambda x: re.findall(r'@(\w+)', x)
)

In [ ]:
df[['tweet','mention']].head(10)


,tweet,mention
0,This cartoon by Alok @caricatured speak about the reality of deals in Indian Politics - Backdoor - or Unholy nexus - when all guys unite on controlling money minting Cricket Administration 😎,[caricatured]
1,I am going to post one #BernieAtTheFarmersProtest every day until either the Farmers protest ends or @BernieSanders @SenSanders notices this and highlights the issue. Or I get blocked by Twitter. #Berniememes #BernieSandersMittens #BernieSanders #FarmersProtest 34 https://t.co/2NFSWxE6fA,"[BernieSanders, SenSanders]"
2,@sardesairajdeep @OfficialUrmila @RahulGandhi So bashing Islamic terrorism is communal but when someone talks about Hindu violence its crappy. Bloody hypocrites. 😪 \n#ModiLaoDeshBanao #ModiHainTohMumkinHain #LokSabhaElections2019 #CongressMuktBharat,"[sardesairajdeep, OfficialUrmila, RahulGandhi]"
3,Oooh... that`s right by the zoo... think... in 2 months` time that could be our regular other meeting place,[]
4,@vivekoberoi @narendramodi @OmungKumar @sureshoberoi @sandip_Ssingh @TSeries @anandpandit63 Absolutely very nice and fantastic Sir 😐❤ Your performance very nice🙋‍♂ Because I am see your movie trailer amazing and fantastic ❤🙋‍♂🙏 #LokSabhaElections2019 #AbkiBaarPhirModiSarkar 🙏,"[vivekoberoi, narendramodi, OmungKumar, sureshoberoi, sandip_Ssingh, TSeries, anandpandit63]"
5,_louise Lucky me. There are mystery ingredients as well,[]
6,డిల్లీ అహంకారంపై .. ఆంధ్రుడి పోరాటం అండగా ఉందాం #narendramodi and #ncbn might have political or personal differences But Modi should not undermine #AndhraPradesh people interests and punish them by stopping project funding #DharmaPorataDeeksha #APDemandsJustice #BeWithBabu,[]
7,Dear @RahulGandhi #RahulGandhi 1 side u offer #pyaarkirajnithi on d other hand u wound an entire community repeatedly #SikhGenocide shows yours n yours party's ethics n committment towards aam jantha.. #NYAYforSikhs #Shame #Congress #HuaTohHua #SamInsultsSikhs,[RahulGandhi]
8,God Bless Ukraine. “The shameful criminalization of freedom of expression must stop.” ~ Marie Struthers https://t.co/j1N1yZ8Sen https://t.co/JDx42B6CgZ #UkraineRussiaWar #Russia #USA #Europe #NATO #China #India #Africa #quote #freedom #freespeech,[]
9,"2/2 Moreover, @kharge has been a tremendous Social reformer, a Great Parliamentarian with a Strong belief in the Constitution of India and Indian secularism ideology with dignity politics 🇮🇳, a loyal warrior of the Congress party @INCIndia @RahulGandhi @priyankagandhi https://t.co/VyJTvNevUf","[kharge, INCIndia, RahulGandhi, priyankagandhi]"


In [ ]:
from collections import Counter

tweet_counts = Counter()
for mentions in df['mention']:
    for name in set(mentions):
        tweet_counts[name] += 1

top_200_tweets = tweet_counts.most_common(200)

top_200_tweets_df = pd.DataFrame(top_200_tweets, columns=['mentions', 'tweet_count'])

total_tweets = len(df)
top_200_tweets_df['percentage'] = (top_200_tweets_df['tweet_count'] / total_tweets * 100).round(2)

print(top_200_tweets_df)

            mentions  tweet_count  percentage
0        RahulGandhi        11157       22.24
1       narendramodi         4199        8.37
2           INCIndia         3609        7.19
3          BJP4India         1744        3.48
4     priyankagandhi          843        1.68
5           PMOIndia          667        1.33
6           AmitShah          619        1.23
7     ArvindKejriwal          545        1.09
8        smritiirani          374        0.75
9           republic          307        0.61
10              ndtv          292        0.58
11               ANI          289        0.58
12        IndiaToday          269        0.54
13          TimesNow          268        0.53
14   sardesairajdeep          267        0.53
15     ShashiTharoor          265        0.53
16    MamataOfficial          260        0.52
17      timesofindia          258        0.51
18       rssurjewala          242        0.48
19     yadavakhilesh          230        0.46
20            aajtak          226 

In [ ]:
import re

no_mentions_df = df[df['mention'].apply(len) == 0]

no_mentions_df = no_mentions_df.copy()
no_mentions_df['hashtags'] = no_mentions_df['tweet'].apply(lambda x: re.findall(r'#(\w+)', x))

total_no_mention_tweets = len(no_mentions_df)
tweets_with_hashtags = (no_mentions_df['hashtags'].apply(len) > 0).sum()
pct_with_hashtags = round(tweets_with_hashtags / total_no_mention_tweets * 100, 2)

print(f"Tweets with no mentions: {total_no_mention_tweets}")
print(f"Of those, tweets with at least one hashtag: {tweets_with_hashtags} ({pct_with_hashtags}%)")

total_hashtag_occurrences = sum(len(h) for h in no_mentions_df['hashtags'])
print(f"Total hashtag occurrences in no-mention tweets: {total_hashtag_occurrences}")

Tweets with no mentions: 28553
Of those, tweets with at least one hashtag: 11532 (40.39%)
Total hashtag occurrences in no-mention tweets: 50321


In [ ]:
from collections import Counter
import re
df['hashtags'] = df['tweet'].astype(str).apply(lambda x: re.findall(r'#(\w+)', x.lower()))
hashtag_counts = Counter(tag for tags in df['hashtags'] for tag in set(tags))

top_200_hashtags = hashtag_counts.most_common(200)
top_200_hashtags_df = pd.DataFrame(top_200_hashtags, columns=['hashtag', 'tweet_count'])

total_tweets = len(df)
top_200_hashtags_df['percentage'] = (top_200_hashtags_df['tweet_count'] / total_tweets * 100).round(2)

pd.set_option('display.max_rows', 200)
print(top_200_hashtags_df)

                    hashtag  tweet_count  percentage
0                     india         9704       19.34
1     loksabhaelections2019         6634       13.22
2               rahulgandhi         5508       10.98
3                       bjp         4373        8.72
4            farmersprotest         2003        3.99
5              narendramodi         1992        3.97
6                  congress         1860        3.71
7                      modi          912        1.82
8                       inc          501        1.00
9             elections2019          445        0.89
10                 pakistan          444        0.89
11                    china          375        0.75
12                  cricket          353        0.70
13                  england          349        0.70
14                     news          341        0.68
15                   indian          334        0.67
16                  wayanad          316        0.63
17               viratkohli          294      

In [ ]:
non_political = set("'arbitrage', 'babarazam', 'bcci', 'bigbrother', 'bitcoin', 'bollywood', 'btc', 'btcinr', 'caatsa', 'chennai', 'consultants', 'covid', 'covid19', 'cricket', 'crickettwitter', 'crypto', 'dalailama', 'development', 'england', 'engvind', 'engvsind', 'food', 'health', 'indiancricketteam', 'indianews', 'indvseng', 'instagramreels', 'latestnews', 'license', 'love', 'maps', 'mapsofindia', 'media', 'monkeypox', 'monkeypoxvirus', 'news18', 'newsupdate', 'odi', 'reels', 'registration', 'rohitsharma', 'sanjusamson', 'sports', 'teamindia', 'tiktok', 'travel', 'twitter', 'viral', 'viratkohli𓃵', 'westindies', 'world', 'worldyouthskillsday', 'म',travel reels love mapsofindia maps crypto म chennai instagramreels caatsa cricket england kingkohli kohli भ latestreels viratkohli𓃵  indveng  coronavirusviratkohli indvseng engvsind engvind crickettwitter odi rohitsharma teamindia bcci indiancricketteam babarazam sports sanjusamson westindies worldcup bitcoin btc btcinr crypto arbitrage monkeypox monkeypoxvirus covid19 covid health bollywood tiktok reels instagramreels bigbrother viral love travel food twitter world maps mapsofindia worldyouthskillsday news18 indianews latestnews media consultants development registration license newsupdate dalailama".split())

political_hashtags_df = top_200_hashtags_df[~top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

removed_df = top_200_hashtags_df[top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

print(f"Kept {len(political_hashtags_df)} political hashtags | Removed {len(removed_df)} non-political")
print(sorted(removed_df['hashtag'].tolist()))


Kept 147 political hashtags | Removed 53 non-political
['arbitrage', 'babarazam', 'bcci', 'bigbrother', 'bitcoin', 'bollywood', 'btc', 'btcinr', 'caatsa', 'chennai', 'consultants', 'covid', 'covid19', 'cricket', 'crickettwitter', 'crypto', 'dalailama', 'development', 'england', 'engvind', 'engvsind', 'food', 'health', 'indiancricketteam', 'indianews', 'indvseng', 'instagramreels', 'latestnews', 'license', 'love', 'maps', 'mapsofindia', 'media', 'monkeypox', 'monkeypoxvirus', 'news18', 'newsupdate', 'odi', 'reels', 'registration', 'rohitsharma', 'sanjusamson', 'sports', 'teamindia', 'tiktok', 'travel', 'twitter', 'viral', 'viratkohli𓃵', 'westindies', 'world', 'worldyouthskillsday', 'म']


In [ ]:
political_counts = [
    (tag, count)
    for tag, count in hashtag_counts.most_common()
    if tag not in non_political
][:200]

top_200_after_removal_df = pd.DataFrame(
    political_counts, columns=['hashtag', 'tweet_count']
)
top_200_after_removal_df['percentage'] = (
    top_200_after_removal_df['tweet_count'] / total_tweets * 100
)

print(f"Top {len(top_200_after_removal_df)} political hashtags after removal")
top_200_after_removal_df

Top 200 political hashtags after removal


,hashtag,tweet_count,percentage
0,india,9704,19.342622
1,loksabhaelections2019,6634,13.223305
2,rahulgandhi,5508,10.978891
3,bjp,4373,8.716538
4,farmersprotest,2003,3.992505
5,narendramodi,1992,3.970579
6,congress,1860,3.707469
7,modi,912,1.817856
8,inc,501,0.998625
9,elections2019,445,0.887002


In [ ]:
print(df.columns.tolist())


['tweet', 'user', 'mention', 'hashtags']


In [ ]:
df['has_mention'] = df['mention'].apply(len) > 0
df['has_hashtag'] = df['hashtags'].apply(len) > 0

at_least_one = df[df['has_mention'] | df['has_hashtag']]

print(f"Tweets with at least one (mention or hashtag): {len(at_least_one)} "
      f"({round(len(at_least_one)/len(df)*100, 2)}%)")

at_least_one[['tweet', 'mention', 'hashtags']]

Tweets with at least one (mention or hashtag): 33148 (66.07%)


,tweet,mention,hashtags
0,This cartoon by Alok @caricatured speak about the reality of deals in Indian Politics - Backdoor - or Unholy nexus - when all guys unite on controlling money minting Cricket Administration 😎,[caricatured],[]
1,I am going to post one #BernieAtTheFarmersProtest every day until either the Farmers protest ends or @BernieSanders @SenSanders notices this and highlights the issue. Or I get blocked by Twitter. #Berniememes #BernieSandersMittens #BernieSanders #FarmersProtest 34 https://t.co/2NFSWxE6fA,"[BernieSanders, SenSanders]","[bernieatthefarmersprotest, berniememes, berniesandersmittens, berniesanders, farmersprotest]"
2,@sardesairajdeep @OfficialUrmila @RahulGandhi So bashing Islamic terrorism is communal but when someone talks about Hindu violence its crappy. Bloody hypocrites. 😪 \n#ModiLaoDeshBanao #ModiHainTohMumkinHain #LokSabhaElections2019 #CongressMuktBharat,"[sardesairajdeep, OfficialUrmila, RahulGandhi]","[modilaodeshbanao, modihaintohmumkinhain, loksabhaelections2019, congressmuktbharat]"
4,@vivekoberoi @narendramodi @OmungKumar @sureshoberoi @sandip_Ssingh @TSeries @anandpandit63 Absolutely very nice and fantastic Sir 😐❤ Your performance very nice🙋‍♂ Because I am see your movie trailer amazing and fantastic ❤🙋‍♂🙏 #LokSabhaElections2019 #AbkiBaarPhirModiSarkar 🙏,"[vivekoberoi, narendramodi, OmungKumar, sureshoberoi, sandip_Ssingh, TSeries, anandpandit63]","[loksabhaelections2019, abkibaarphirmodisarkar]"
6,డిల్లీ అహంకారంపై .. ఆంధ్రుడి పోరాటం అండగా ఉందాం #narendramodi and #ncbn might have political or personal differences But Modi should not undermine #AndhraPradesh people interests and punish them by stopping project funding #DharmaPorataDeeksha #APDemandsJustice #BeWithBabu,[],"[narendramodi, ncbn, andhrapradesh, dharmaporatadeeksha, apdemandsjustice, bewithbabu]"
...,...,...,...
50162,@OmarAbdullah @RahulGandhi So who wants #NarendraModi as Prime Minister after #Elections2019 ?\nIs it #ImranKhan or someone more powerful than these two ?\nNow question yourself that was #BalakotAirStrike &amp; #PulwamaTerrorAttack planned by HIM ? \nThe answer my friend is blowing in the wind.,"[OmarAbdullah, RahulGandhi]","[narendramodi, elections2019, imrankhan, balakotairstrike, pulwamaterrorattack]"
50164,ROYALICA Women Black Georgette Anarkali Kurta Palazzo Set with Dupatta (X-Large) https://t.co/D6pp1yQYLn #India #USA #clothing https://t.co/zmDFZGdKjx,[],"[india, usa, clothing]"
50165,#Latamangeshkar is one of the star campaigners of #Bjp for #LokSabhaElections2019 and strong supporter of @narendramodi So many of my idols have fallen from grace! Support for BJP is truly a litmus test of exposing one's value systems! Sorry @mangeshkarlata u disappointed 😞,"[narendramodi, mangeshkarlata]","[latamangeshkar, bjp, loksabhaelections2019]"
50167,@RahulGandhi @RahulGandhi Shame on you and your congress party. Congressman B K HariPrasad is using very bad words for Sh. @AmitShah who is suffering from Swine Flu. Your party will never come in power @RahulGandhi #Pappu\n\nJust Shameless.\n\n#BKHariprasad \n#Shame\n#RahulGandhi,"[RahulGandhi, RahulGandhi, AmitShah, RahulGandhi]","[pappu, bkhariprasad, shame, rahulgandhi]"


In [ ]:
STOPWORDS = set(stopwords.words('english'))

HINDI_SW = {
    'hai','hain','bhi','ka','ki','ke','ko','se','aur','main','mein',
    'nahi','aap','ho','toh','ye','yeh','wo','woh','ne','pe','kya',
    'tha','thi','kar','liye','phir','ab','ek','do','teen'
}

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split()
              if w not in STOPWORDS and w not in HINDI_SW and len(w) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['tweet'].apply(preprocess)
df = df.drop_duplicates(subset=['tweet']).reset_index(drop=True)
df = df[df['clean_text'].str.split().str.len() >= 3].reset_index(drop=True)

print(f"After preprocessing : {len(df):,} tweets")
print(f"\nBefore: {df['tweet'].iloc[2][:200]}")
print(f"After : {df['clean_text'].iloc[2][:200]}")

After preprocessing : 47,990 tweets

Before: @sardesairajdeep @OfficialUrmila @RahulGandhi So bashing Islamic terrorism is communal but when someone talks about Hindu violence its crappy.  Bloody hypocrites. 😪 
#ModiLaoDeshBanao #ModiHainTohMumk
After : sardesairajdeep officialurmila rahulgandhi bashing islamic terrorism communal someone talks hindu violence crappy bloody hypocrites modilaodeshbanao modihaintohmumkinhain loksabhaelections congressmuk


In [ ]:
!pip install textblob --break-system-packages
!python -m textblob.download_corpora


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.


In [ ]:
tfidf   = TfidfVectorizer(max_features=5000, ngram_range=(1,1),
                           min_df=3, max_df=0.90, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df['clean_text'])

vocab       = tfidf.get_feature_names_out()
mean_scores = np.asarray(X_tfidf.mean(axis=0)).flatten()
doc_freq    = np.asarray((X_tfidf > 0).sum(axis=0)).flatten()
total_docs  = X_tfidf.shape[0]

top_1000_idx    = mean_scores.argsort()[-1000:][::-1]
sample_words  = [vocab[i] for i in top_1000_idx]
top_1000_scores = [float(mean_scores[i]) for i in top_1000_idx]
top_1000_dpct   = [float(doc_freq[i]/total_docs*100) for i in top_1000_idx]

print(f"TF-IDF full vocabulary : {len(vocab)}")
print(f"Top 1000 extracted     : 1,000")
for i in range(50):
    print(f"  {sample_words[i]:<20} score={top_1000_scores[i]:.5f}  doc%={top_1000_dpct[i]:.1f}%")


TF-IDF full vocabulary : 5000
Top 1000 extracted     : 1,000
  india                score=0.03698  doc%=24.9%
  rahulgandhi          score=0.03603  doc%=26.7%
  indian               score=0.02103  doc%=13.8%
  politics             score=0.01980  doc%=12.6%
  loksabhaelections    score=0.01895  doc%=13.8%
  narendramodi         score=0.01839  doc%=11.6%
  bjp                  score=0.01786  doc%=11.7%
  modi                 score=0.01365  doc%=7.9%
  congress             score=0.01287  doc%=7.6%
  incindia             score=0.01257  doc%=7.7%
  amp                  score=0.01226  doc%=7.3%
  gandhi               score=0.01178  doc%=7.0%
  rahul                score=0.01155  doc%=6.7%
  like                 score=0.01055  doc%=5.3%
  farmersprotest       score=0.01003  doc%=3.9%
  day                  score=0.00997  doc%=3.8%
  dont                 score=0.00958  doc%=4.6%
  one                  score=0.00913  doc%=4.4%
  people               score=0.00881  doc%=4.7%
  get               

In [ ]:

NLTK_STOPWORDS = set(stopwords.words('english'))

DROP_FROM_VOCAB = {
    'good','bad','great','best','worst','nice','better','well',
    'really','very','much','many','every','even','just','like',
    'also','still','back','right','left','now','only','would',
    'could','should','want','think','know','say','said','see',
    'get','got','make','made','give','take','come','going',
    'amp','people','like','dont','one','know','team',
    'good','even','time','country','see','get','new','also',
    'never','think','would','man','much','every','many','world',

    # ── generic people/time/place words ───────────────────────
    'man','men','woman','women','people','person','time','day',
    'year','week','month','today','yesterday','tomorrow','way',
    'thing','things','place','world','country','city','state',
    'home','house','life','hand','eye','face','name','number',

    # ── common verbs (no political meaning) ───────────────────
    'said','told','asked','went','came','put','let','run','ran',
    'used','using','done','doing','look','looking','looked',
    'work','worked','working','help','helped','helping','try',
    'tried','trying','seem','seems','seemed','feel','felt',
    'show','showed','shown','keep','kept','call','called',
    'talk','talked','talking','write','wrote','written','read',
        'follow','followed','share','shared','sharing','post','posted',

    # ── twitter/social media artifacts ────────────────────────
    'amp','via','co','http','https','rt','please','need',
    'follow','retweet','tweet','tweeted','thread','account',
    'handle','profile','click','link','watch','read','check',

    # ── generic quantity/frequency words ──────────────────────
    'never','always','ever','already','yet','still','else',
    'more','less','most','least','few','many','lot','lots',
    'enough','enough','whole','full','half','part','bit',
    'some','any','all','both','each','every','either',
    'another','other','others','same','different','last','next',

    # ── generic adjectives (appear everywhere) ────────────────
    'new','old','big','small','large','long','short','high',
    'low','early','late','first','second','last','important',
    'real','true','false','wrong','right','strong','weak',
    'hard','easy','free','open','close','clear','common',
    'public','private','official','special','major','minor',

    # ── generic news/reporting language ───────────────────────
    'news','report','reported','reporting','says','claim',
    'claimed','statement','source','sources','according',
    'based','given','noted','added','confirmed','denied',
    'announced','announced','mentioned','revealed','stated',

    # ── common connector/filler words ─────────────────────────
    'also','however','therefore','thus','hence','though',
    'although','because','since','while','when','where',
    'which','whose','whom','whether','instead','despite',
    'against','towards','within','without','across','through',
    'during','before','after','above','below','between',
    'among','around','behind','beside','beyond','inside',

    # ── numbers and generic quantifiers ───────────────────────
    'one','two','three','four','five','six','seven','eight',
    'nine','ten','hundred','thousand','million','billion',
    'crore','lakh',

    # ── common Indian-English generic words ───────────────────
    'ji','sir','dear','respected','brother','sister','friend',
    'bhai','didi','agar','phir','bas','karo','karke','abhi',
    'wala','wali','wale','please','kindly','request','thanks',
    'thank','welcome','congrats','sorry','okay','yes','yeah',
    'nope','haha','lol','omg','wow','oh','ah','hmm',

    # ── generic action/state words ────────────────────────────
    'start','started','starting','stop','stopped','end',
    'ended','begin','began','continue','continued','change',
    'changed','move','moved','stand','stood','stay','stayed',
    'live','lived',

    # ── very common Indian political dataset noise ────────────
    # ── generic verbs ────────────────────────────────────────
    'need', 'great', 'years', 'well', 'want', 'say', 'come',
    'make', 'best', 'take', 'cant', 'understand', 'become',
    'give', 'made', 'play', 'work', 'keep', 'real', 'using',
    'used', 'said', 'since', 'bad', 'without', 'seen',
    'getting', 'let', 'got', 'yes', 'done', 'read', 'talk',
    'called', 'look', 'show', 'find', 'ask', 'put', 'run',
    'happen', 'call', 'shows', 'leave', 'create', 'stay',
    'start', 'bring', 'try', 'follow', 'speak', 'tell',

    # ── generic adjectives ───────────────────────────────────
    'thats', 'better', 'may', 'big', 'doesnt', 'really',
    'real', 'worst', 'happy', 'long', 'dirty', 'true',
    'hard', 'strong', 'greatest', 'ultimate', 'unbeatable',
    'interesting', 'full', 'low', 'common', 'entire', 'huge',
    'completely', 'clear', 'young', 'sure', 'different',
    'wrong', 'high', 'least', 'possible', 'next', 'first',
    'always', 'nothing', 'ever', 'please', 'today', 'sir',

    # ── generic nouns ────────────────────────────────────────
    'person', 'day', 'thing', 'family', 'name', 'life',
    'level', 'game', 'things', 'two', 'etc', 'way', 'year',
    'times', 'moment', 'sense', 'idea', 'point', 'chance',
    'words', 'side', 'place', 'back', 'fact', 'days', 'lot',
    'part', 'birthday', 'book', 'story', 'job', 'example',
    'difference', 'birth', 'home', 'mind', 'thought',

    # ── generic connectors/fillers ────────────────────────────
    'going', 'still', 'away', 'making', 'yet', 'must',
    'already', 'based', 'towards', 'around', 'coming',
    'though', 'never', 'else', 'rather', 'beyond', 'past',
    'last', 'behind', 'general', 'whatever', 'single',
    'others', 'cannot', 'among', 'without', 'instead',
    'definitely', 'completely', 'especially', 'following',
    'actually', 'someone', 'anyone', 'everyone', 'something',
    'everything', 'anything', 'nothing', 'enough', 'till',

    # ── twitter noise ────────────────────────────────────────
    'twitter', 'tweet', 'via', 'lol', 'ppl', 'hes', 'youre',
    'isnt', 'didnt', 'wont', 'thats', 'whats', 'dont',
    'cant', 'doesnt', 'lets', 'youll', 'its',

    # ── sports/entertainment (non-political) ─────────────────
    'cricket', 'bcci', 'sports', 'team', 'player', 'players',
    'playing', 'played', 'film', 'movie', 'bollywood', 'cinema',
    'fans', 'star', 'sanju', 'game', 'play',

    # ── generic sentiments (no political specificity) ─────────
    'happy', 'sad', 'love', 'hope', 'wish', 'wishes',
    'sorry', 'welcome', 'thanks', 'congratulations', 'dear',
    'care', 'feel', 'feeling', 'respect', 'proud',

    # ── common people references ─────────────────────────────
    'man', 'men', 'woman', 'women', 'guy', 'guys', 'lady',
    'king', 'hero', 'person', 'human',

    # ── generic misc ─────────────────────────────────────────
    'matter', 'matters', 'remain', 'seems', 'relevant',
    'irrelevant', 'understanding', 'become', 'use', 'self',
    'origin', 'role', 'black', 'white', 'old', 'post',
    'watch', 'watching', 'global', 'learn', 'reason', 'agree',
    'born', 'means', 'makes', 'join', 'forget', 'comment',
    'involved', 'start', 'face', 'top', 'position',
}

ALL_STOP = NLTK_STOPWORDS | DROP_FROM_VOCAB

MIN_WORD_LEN = 3
MAX_DOC_PCT  = 60.0

political_vocab = []
removed_vocab   = []

for word, score, dpct in zip(sample_words, top_1000_scores, top_1000_dpct):
    if word.lower() in ALL_STOP:
        removed_vocab.append((word, score, dpct, 'stopword/no political meaning')); continue
    if len(word) < MIN_WORD_LEN:
        removed_vocab.append((word, score, dpct, 'too short')); continue
    if dpct > MAX_DOC_PCT:
        removed_vocab.append((word, score, dpct, f'in {dpct:.0f}% docs')); continue
    political_vocab.append((word, score, dpct))







In [ ]:
print(f"After filtering  : {len(political_vocab):,} words")
print(f"Removed          : {len(DROP_FROM_VOCAB ):,} words")
print()
for i, (w, s, d) in enumerate(political_vocab[:50]):
    print(f"  {i+1:<4} {w:<25} score={s:.5f}  doc%={d:.1f}%")

After filtering  : 675 words
Removed          : 478 words

  1    india                     score=0.03698  doc%=24.9%
  2    rahulgandhi               score=0.03603  doc%=26.7%
  3    indian                    score=0.02103  doc%=13.8%
  4    politics                  score=0.01980  doc%=12.6%
  5    loksabhaelections         score=0.01895  doc%=13.8%
  6    narendramodi              score=0.01839  doc%=11.6%
  7    bjp                       score=0.01786  doc%=11.7%
  8    modi                      score=0.01365  doc%=7.9%
  9    congress                  score=0.01287  doc%=7.6%
  10   incindia                  score=0.01257  doc%=7.7%
  11   gandhi                    score=0.01178  doc%=7.0%
  12   rahul                     score=0.01155  doc%=6.7%
  13   farmersprotest            score=0.01003  doc%=3.9%
  14   bjpindia                  score=0.00745  doc%=4.0%
  15   party                     score=0.00628  doc%=3.1%
  16   vote                      score=0.00626  doc%=2.7%
  17  

In [ ]:
clean_vocab_list = [w for w, s, d in political_vocab[:700]]

tfidf_pol = TfidfVectorizer(
    vocabulary    = clean_vocab_list,  # restrict to cleaned vocab only
    sublinear_tf  = True
)

X_pol_only    = tfidf_pol.fit_transform(df['clean_text'])
political_cols = list(range(X_pol_only.shape[1]))   # all cols = political cols

political_score = np.asarray(X_pol_only.sum(axis=1)).flatten()
df['political_score'] = political_score

non_zero  = political_score[political_score > 0]
threshold = float(np.median(non_zero))
df['is_political'] = (df['political_score'] > 0).astype(int)

pol_count    = int(df['is_political'].sum())
nonpol_count = int((df['is_political']==0).sum())

print(f"Vocabulary used              : {len(clean_vocab_list)}")
print(f"Identified as POLITICAL      : {pol_count:,}  ({pol_count/len(df)*100:.1f}%)")
print(f"Identified as NON-POLITICAL  : {nonpol_count:,}  ({nonpol_count/len(df)*100:.1f}%)")
print()
print("top  words used:")
for w in clean_vocab_list[:1000]:
    print(f"  {w}")

print()


Vocabulary used              : 675
Identified as POLITICAL      : 44,336  (92.4%)
Identified as NON-POLITICAL  : 3,654  (7.6%)

top  words used:
  india
  rahulgandhi
  indian
  politics
  loksabhaelections
  narendramodi
  bjp
  modi
  congress
  incindia
  gandhi
  rahul
  farmersprotest
  bjpindia
  party
  vote
  elections
  election
  pakistan
  priyankagandhi
  farmers
  win
  support
  nation
  leader
  power
  delhi
  pmoindia
  govt
  amitshah
  mothers
  political
  government
  pappu
  night
  minister
  arvindkejriwal
  indias
  president
  inc
  money
  china
  media
  amethi
  miss
  kerala
  morning
  wayanad
  england
  shame
  hate
  fun
  prime
  national
  poor
  leaders
  ill
  soon
  smritiirani
  wait
  hindu
  wants
  fight
  case
  gonna
  history
  biggest
  future
  seats
  contest
  indians
  lost
  namoagain
  democracy
  usa
  opposition
  tonight
  food
  school
  viratkohli
  uae
  covid
  corruption
  weekend
  sleep
  rafale
  ani
  court
  ndtv
  needs

In [ ]:

from collections import Counter as _Counter
_mention_counter = _Counter(m for ms in df['mention'] for m in ms)
_hashtag_counter = _Counter(h for hs in df['hashtags'] for h in hs)
political_mentions = {m for m, _ in _mention_counter.most_common(200)}
political_hashtags = {h for h, _ in _hashtag_counter.most_common(200)}
political_mentions = {m.lower() for m in political_mentions}
political_hashtags = {h.lower() for h in political_hashtags}

MENTION_WEIGHT = 1.0
HASHTAG_WEIGHT = 1.0
CAP = 3

def political_mention_hashtag_score(mentions, hashtags):
    mention_hits = sum(1 for m in mentions if m.lower() in political_mentions)
    hashtag_hits = sum(1 for h in hashtags if h.lower() in political_hashtags)

    capped_mention_hits = min(mention_hits, CAP)
    capped_hashtag_hits = min(hashtag_hits, CAP)

    score = (capped_mention_hits * MENTION_WEIGHT) + (capped_hashtag_hits * HASHTAG_WEIGHT)
    has_signal = int((mention_hits > 0) or (hashtag_hits > 0))

    return pd.Series({
        'mention': mention_hits,
        'hashtag': hashtag_hits,
        'pol_score': score,
        'has_political_signal_mh': has_signal
    })

df[['political_mention_hits', 'political_hashtag_hits',
    'political_score_mh', 'has_political_signal_mh']] = df.apply(
    lambda row: political_mention_hashtag_score(row['mention'], row['hashtags']), axis=1
)

print("Both mention+hashtag hit:", len(df[(df['political_mention_hits'] > 0) & (df['political_hashtag_hits'] > 0)]))
print("Only mention hit:        ", len(df[(df['political_mention_hits'] > 0) & (df['political_hashtag_hits'] == 0)]))
print("Only hashtag hit:        ", len(df[(df['political_mention_hits'] == 0) & (df['political_hashtag_hits'] > 0)]))
print("Neither:                 ", len(df[(df['political_mention_hits'] == 0) & (df['political_hashtag_hits'] == 0)]))

df[['tweet', 'political_mention_hits', 'political_hashtag_hits',
    'political_score_mh', 'has_political_signal_mh']].head(10)

Both mention+hashtag hit: 14964
Only mention hit:         1606
Only hashtag hit:         13371
Neither:                  18049


,tweet,political_mention_hits,political_hashtag_hits,political_score_mh,has_political_signal_mh
0,This cartoon by Alok @caricatured speak about the reality of deals in Indian Politics - Backdoor - or Unholy nexus - when all guys unite on controlling money minting Cricket Administration 😎,0.0,0.0,0.0,0.0
1,I am going to post one #BernieAtTheFarmersProtest every day until either the Farmers protest ends or @BernieSanders @SenSanders notices this and highlights the issue. Or I get blocked by Twitter. #Berniememes #BernieSandersMittens #BernieSanders #FarmersProtest 34 https://t.co/2NFSWxE6fA,0.0,1.0,1.0,1.0
2,@sardesairajdeep @OfficialUrmila @RahulGandhi So bashing Islamic terrorism is communal but when someone talks about Hindu violence its crappy. Bloody hypocrites. 😪 \n#ModiLaoDeshBanao #ModiHainTohMumkinHain #LokSabhaElections2019 #CongressMuktBharat,2.0,2.0,4.0,1.0
3,Oooh... that`s right by the zoo... think... in 2 months` time that could be our regular other meeting place,0.0,0.0,0.0,0.0
4,@vivekoberoi @narendramodi @OmungKumar @sureshoberoi @sandip_Ssingh @TSeries @anandpandit63 Absolutely very nice and fantastic Sir 😐❤ Your performance very nice🙋‍♂ Because I am see your movie trailer amazing and fantastic ❤🙋‍♂🙏 #LokSabhaElections2019 #AbkiBaarPhirModiSarkar 🙏,1.0,2.0,3.0,1.0
5,_louise Lucky me. There are mystery ingredients as well,0.0,0.0,0.0,0.0
6,డిల్లీ అహంకారంపై .. ఆంధ్రుడి పోరాటం అండగా ఉందాం #narendramodi and #ncbn might have political or personal differences But Modi should not undermine #AndhraPradesh people interests and punish them by stopping project funding #DharmaPorataDeeksha #APDemandsJustice #BeWithBabu,0.0,1.0,1.0,1.0
7,Dear @RahulGandhi #RahulGandhi 1 side u offer #pyaarkirajnithi on d other hand u wound an entire community repeatedly #SikhGenocide shows yours n yours party's ethics n committment towards aam jantha.. #NYAYforSikhs #Shame #Congress #HuaTohHua #SamInsultsSikhs,1.0,2.0,3.0,1.0
8,God Bless Ukraine. “The shameful criminalization of freedom of expression must stop.” ~ Marie Struthers https://t.co/j1N1yZ8Sen https://t.co/JDx42B6CgZ #UkraineRussiaWar #Russia #USA #Europe #NATO #China #India #Africa #quote #freedom #freespeech,0.0,4.0,3.0,1.0
9,"2/2 Moreover, @kharge has been a tremendous Social reformer, a Great Parliamentarian with a Strong belief in the Constitution of India and Indian secularism ideology with dignity politics 🇮🇳, a loyal warrior of the Congress party @INCIndia @RahulGandhi @priyankagandhi https://t.co/VyJTvNevUf",4.0,0.0,3.0,1.0


In [ ]:
import re
import pandas as pd
if 'mention' not in df.columns:
    df['mention'] = df['tweet'].astype(str).apply(lambda x: [m.lower() for m in re.findall(r'@(\w+)', x)])

if 'hashtags' not in df.columns:
    df['hashtags'] = df['tweet'].astype(str).apply(lambda x: [h.lower() for h in re.findall(r'#(\w+)', x)])

df['has_mention'] = df['mention'].apply(len) > 0
df['has_hashtag'] = df['hashtags'].apply(len) > 0
at_least_one = df[df['has_mention'] | df['has_hashtag']]

total_tweets = len(df)
n_with_mention = df['has_mention'].sum()
n_with_hashtag = df['has_hashtag'].sum()
n_with_either = len(at_least_one)

print("STEP 1 — Extraction verification")
print(f"Total tweets                 : {total_tweets:,}")
print(f"Tweets with mentions         : {n_with_mention:,} ({n_with_mention/total_tweets*100:.2f}%)")
print(f"Tweets with hashtags         : {n_with_hashtag:,} ({n_with_hashtag/total_tweets*100:.2f}%)")
print(f"Tweets with either           : {n_with_either:,} ({n_with_either/total_tweets*100:.2f}%)")

df[['tweet', 'mention', 'hashtags']].head(5)




mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 2 — Mention frequency table (top 20)")
mention_freq_df.head(20)


hashtag_freq = df['hashtags'].apply(set).explode().value_counts()
hashtag_freq_df = hashtag_freq.reset_index()
hashtag_freq_df.columns = ['hashtag', 'frequency']
hashtag_freq_df = hashtag_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 3 — Hashtag frequency table (top 20)")
hashtag_freq_df.head(20)




total_unique_mentions = mention_freq_df['mention'].nunique()
total_unique_hashtags = hashtag_freq_df['hashtag'].nunique()
total_mention_occurrences = mention_freq_df['frequency'].sum()
total_hashtag_occurrences = hashtag_freq_df['frequency'].sum()

print("\nSTEP 4 — Frequency analysis")
print(f"Total unique mentions     : {total_unique_mentions:,}")
print(f"Total unique hashtags     : {total_unique_hashtags:,}")
print(f"Total mention occurrences : {total_mention_occurrences:,}")
print(f"Total hashtag occurrences : {total_hashtag_occurrences:,}")

for n in [50, 100, 200, 500]:
    top_n_mentions = set(mention_freq_df['mention'].head(n))
    top_n_hashtags = set(hashtag_freq_df['hashtag'].head(n))

    cov_mention = df['mention'].apply(lambda names: bool(set(names) & top_n_mentions)).sum()
    cov_hashtag = df['hashtags'].apply(lambda tags: bool(set(tags) & top_n_hashtags)).sum()

    print(f"\nTop {n} mentions cover : {cov_mention:,} tweets ({cov_mention/total_tweets*100:.2f}%)")
    print(f"Top {n} hashtags cover : {cov_hashtag:,} tweets ({cov_hashtag/total_tweets*100:.2f}%)")

political_mention_labeling_table = mention_freq_df.copy()
political_mention_labeling_table['Label'] = ""

political_hashtag_labeling_table = hashtag_freq_df.copy()
political_hashtag_labeling_table['Label'] = ""

print("\nSTEP 5 — Labeling tables ready")
political_mention_labeling_table.head(20)
political_hashtag_labeling_table.head(20)
political_mention_labeling_table.to_csv('political_mention_labeling_table.csv', index=False)
political_hashtag_labeling_table.to_csv('political_hashtag_labeling_table.csv', index=False)

STEP 1 — Extraction verification
Total tweets                 : 47,990
Tweets with mentions         : 21,504 (44.81%)
Tweets with hashtags         : 29,191 (60.83%)
Tweets with either           : 32,742 (68.23%)

STEP 2 — Mention frequency table (top 20)

STEP 3 — Hashtag frequency table (top 20)

STEP 4 — Frequency analysis
Total unique mentions     : 12,065
Total unique hashtags     : 20,180
Total mention occurrences : 61,287
Total hashtag occurrences : 107,976

Top 50 mentions cover : 15,294 tweets (31.87%)
Top 50 hashtags cover : 28,220 tweets (58.80%)

Top 100 mentions cover : 15,815 tweets (32.95%)
Top 100 hashtags cover : 28,260 tweets (58.89%)

Top 200 mentions cover : 16,557 tweets (34.50%)
Top 200 hashtags cover : 28,335 tweets (59.04%)

Top 500 mentions cover : 17,520 tweets (36.51%)
Top 500 hashtags cover : 28,397 tweets (59.17%)

STEP 5 — Labeling tables ready


In [ ]:
import re
import pandas as pd
from IPython.display import display

if 'mention' not in df.columns:
    df['mention'] = df['tweet'].astype(str).apply(lambda x: [m.lower() for m in re.findall(r'@(\w+)', x)])

if 'hashtags' not in df.columns:
    df['hashtags'] = df['tweet'].astype(str).apply(lambda x: [h.lower() for h in re.findall(r'#(\w+)', x)])

df['has_mention'] = df['mention'].apply(len) > 0
df['has_hashtag'] = df['hashtags'].apply(len) > 0
at_least_one = df[df['has_mention'] | df['has_hashtag']]

print(f"Tweets with at least one (mention or hashtag): {len(at_least_one)} "
      f"({round(len(at_least_one)/len(df)*100, 2)}%)")
at_least_one[['tweet', 'mention', 'hashtags']].to_csv('tweets_with_mention_or_hashtag.csv', index=False)
print(f"Saved {len(at_least_one):,} tweets to tweets_with_mention_or_hashtag.csv")

mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 2 — Mention frequency table ", len(mention_freq_df), "unique mentions)")
display(mention_freq_df.head(200))

hashtag_freq = df['hashtags'].apply(set).explode().value_counts()
hashtag_freq_df = hashtag_freq.reset_index()
hashtag_freq_df.columns = ['hashtag', 'frequency']
hashtag_freq_df = hashtag_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 3 — Hashtag frequency table (top 20 of", len(hashtag_freq_df), "unique hashtags)")
display(hashtag_freq_df.head(200))






Tweets with at least one (mention or hashtag): 32742 (68.23%)
Saved 32,742 tweets to tweets_with_mention_or_hashtag.csv

STEP 2 — Mention frequency table  12065 unique mentions)


,mention,frequency
0,RahulGandhi,11156
1,narendramodi,4196
2,INCIndia,3609
3,BJP4India,1743
4,priyankagandhi,843
5,PMOIndia,658
6,AmitShah,614
7,ArvindKejriwal,541
8,smritiirani,374
9,republic,305



STEP 3 — Hashtag frequency table (top 20 of 20180 unique hashtags)


,hashtag,frequency
0,india,9456
1,loksabhaelections2019,6634
2,rahulgandhi,5507
3,bjp,4373
4,narendramodi,1990
5,farmersprotest,1864
6,congress,1860
7,modi,908
8,inc,501
9,elections2019,445


In [ ]:
# Create top_200_mentions_df from tweet_counts
top_200_mentions_df = pd.DataFrame(tweet_counts.most_common(200), columns=['mention', 'tweet_count'])
total_tweets = len(top_200_mentions_df)
top_200_mentions_df['percentage'] = (top_200_mentions_df['tweet_count'] / total_tweets * 100).round(2)

# Define non_political mentions set
non_political = set("""bcci imvkohli youtube imro45 WithPGV akshaykumar twitter elonmusk IamSanjuSamson BCCI imVkohli
rihanna AnupamPKher OxfordWords iamsrk vivekoberoi ESPNcricinfo ICC CNN Indian_Analyzer
BBCWorld babarazam258 khushsundar ashutosh83B kamaalkhan vikrantgupta73 DeepSandhu_K _YogendraYadav Tractor2twitr Twitter""".split())

political_mentions_df = top_200_mentions_df[~top_200_mentions_df['mention'].isin(non_political)].reset_index(drop=True)

removed_df = top_200_mentions_df[top_200_mentions_df['mention'].isin(non_political)].reset_index(drop=True)

print(f"Kept {len(political_mentions_df)} political mentions | Removed {len(removed_df)} non-political")
print(sorted(removed_df['mention'].tolist()))

Kept 189 political mentions | Removed 11 non-political
['BCCI', 'DeepSandhu_K', 'IamSanjuSamson', 'Indian_Analyzer', 'Tractor2twitr', 'Twitter', 'WithPGV', 'akshaykumar', 'ashutosh83B', 'elonmusk', 'imVkohli']


In [ ]:
mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\n Mention frequency table ", len(mention_freq_df), "unique mentions)")
display(mention_freq_df.head(300))


 Mention frequency table  12065 unique mentions)


,mention,frequency
0,RahulGandhi,11156
1,narendramodi,4196
2,INCIndia,3609
3,BJP4India,1743
4,priyankagandhi,843
...,...,...
295,eminem,18
296,ani_digital,18
297,UNHumanRights,18
298,SirJadeja,18


In [ ]:
non_political = set("""
arbitrage babarazam bcci bigbrother bitcoin bollywood btc btcinr caatsa chennai
consultants covid covid19 cricket crickettwitter crypto dalailama development
england engvind engvsind food health indiancricketteam indianews indvseng
instagramreels latestnews license love maps mapsofindia media monkeypox
monkeypoxvirus news18 newsupdate odi reels registration rohitsharma sanjusamson
sports teamindia tiktok travel twitter viral viratkohli𓃵 westindies world
worldyouthskillsday म chennai kingkohli kohli भ latestreels indveng
coronavirusviratkohli reelsindia tiktoks worldcup,peace,jaspritbumrah, ipl2019, kingkohli, kohli, maps, viratkohli𓃵
""".split())

political_hashtags_df = top_200_hashtags_df[~top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

removed_df = top_200_hashtags_df[top_200_hashtags_df['hashtag'].isin(non_political)].reset_index(drop=True)

print(f"Kept {len(political_hashtags_df)} political hashtags | Removed {len(removed_df)} non-political")
print(sorted(removed_df['hashtag'].tolist()))

Kept 147 political hashtags | Removed 53 non-political
['arbitrage', 'babarazam', 'bcci', 'bigbrother', 'bitcoin', 'bollywood', 'btc', 'btcinr', 'caatsa', 'chennai', 'consultants', 'covid', 'covid19', 'cricket', 'crickettwitter', 'crypto', 'dalailama', 'development', 'england', 'engvind', 'engvsind', 'food', 'health', 'indiancricketteam', 'indianews', 'indvseng', 'instagramreels', 'latestnews', 'license', 'love', 'maps', 'mapsofindia', 'media', 'monkeypox', 'monkeypoxvirus', 'news18', 'newsupdate', 'odi', 'reels', 'registration', 'rohitsharma', 'sanjusamson', 'sports', 'teamindia', 'tiktok', 'travel', 'twitter', 'viral', 'viratkohli𓃵', 'westindies', 'world', 'worldyouthskillsday', 'म']


In [ ]:
import re

hashtag_freq = df['hashtags'].apply(set).explode().value_counts()
hashtag_freq_df = hashtag_freq.reset_index()
hashtag_freq_df.columns = ['hashtag', 'frequency']
hashtag_freq_df = hashtag_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)

print("\nSTEP 3 - Hashtag frequency table ", len(hashtag_freq_df), "unique hashtags)")
display(hashtag_freq_df.head(300))

# ──── non_political exclusion set, fixed to split on both commas and whitespace ────
non_political_hashtags_new = set(""" ukrainerussiawar russia usa europe nato china india africa quote freedom freespeech corona coronavirus boosterdose vaccines coronavaccines dainiikgomantak dainiikgomantaknews  म ज घर_व babarazam viratkohli crickettwitter icc oscar msp_च greenenergy cleanenergy hydrogen fuelcell windpower solarenergy renewableenergy cleanseas renewables climatechange morocco climatecrisis plasticwaste pcn piracy bisleri monkeypox thiruvananthapuram uae """.split())

political_hashtags_df = hashtag_freq_df[~hashtag_freq_df['hashtag'].isin(non_political_hashtags_new)].reset_index(drop=True)

removed_df = hashtag_freq_df[hashtag_freq_df['hashtag'].isin(non_political_hashtags_new)].reset_index(drop=True)

print(f"Kept {len(political_hashtags_df)} political hashtags | Removed {len(removed_df)} non-political")
print(sorted(removed_df['hashtag'].tolist()))


STEP 3 - Hashtag frequency table  20180 unique hashtags)


,hashtag,frequency
0,india,9456
1,loksabhaelections2019,6634
2,rahulgandhi,5507
3,bjp,4373
4,narendramodi,1990
...,...,...
295,raulvinci,35
296,pmo,35
297,live,34
298,adani,34


Kept 20134 political hashtags | Removed 46 non-political
['africa', 'babarazam', 'bisleri', 'boosterdose', 'china', 'cleanenergy', 'cleanseas', 'climatechange', 'climatecrisis', 'corona', 'coronavaccines', 'coronavirus', 'crickettwitter', 'dainiikgomantak', 'dainiikgomantaknews', 'europe', 'freedom', 'freespeech', 'fuelcell', 'greenenergy', 'hydrogen', 'icc', 'india', 'monkeypox', 'morocco', 'msp_च', 'nato', 'oscar', 'pcn', 'piracy', 'plasticwaste', 'quote', 'renewableenergy', 'renewables', 'russia', 'solarenergy', 'thiruvananthapuram', 'uae', 'ukrainerussiawar', 'usa', 'vaccines', 'viratkohli', 'windpower', 'घर_व', 'ज', 'म']


In [ ]:
top_300_political_hashtags = set(political_hashtags_df.head(300)['hashtag'])

df['has_top300_political_hashtag'] = df['hashtags'].apply(lambda tags: bool(set(tags) & top_300_political_hashtags))

n_covered_300 = df['has_top300_political_hashtag'].sum()
total_tweets = len(df)

print(f"Total tweets                                : {total_tweets:,}")
print(f" top-300 political hashtag: {n_covered_300:,} ({n_covered_300/total_tweets*100:.2f}%)")

Total tweets                                : 47,990
 top-300 political hashtag: 25,181 (52.47%)


In [ ]:
top_200_political_hashtags = set(political_hashtags_df.head(200)['hashtag'])

df['has_top200_political_hashtag'] = df['hashtags'].apply(lambda tags: bool(set(tags) & top_200_political_hashtags))

n_covered = df['has_top200_political_hashtag'].sum()
total_tweets = len(df)

print(f" tweets                                : {total_tweets:,}")
print(f" top-200 political hashtag TWEETS: {n_covered:,} ({n_covered/total_tweets*100:.2f}%)")

 tweets                                : 47,990
 top-200 political hashtag TWEETS: 24,604 (51.27%)


In [ ]:
# ============================================================
# STEP 4 - Entity-level binary label dictionaries
# Political entities (top-200 mentions + top-200 hashtags) -> 1
# Non-political entities (removed via non_political filter) -> 0
# NOTE: entity-level labels only (not tweet labels)
# ============================================================
import pandas as pd
#
# 1) Build the MENTION frequency table (mirrors hashtag_freq_df)
mention_freq = df['mention'].apply(set).explode().value_counts()
mention_freq_df = mention_freq.reset_index()
mention_freq_df.columns = ['mention', 'frequency']
mention_freq_df = mention_freq_df.sort_values('frequency', ascending=False).reset_index(drop=True)
#
political_mentions_df = mention_freq_df[~mention_freq_df['mention'].isin(non_political)].reset_index(drop=True)
removed_mentions_df = mention_freq_df[mention_freq_df['mention'].isin(non_political)].reset_index(drop=True)
print(f"Kept {len(political_mentions_df)} political mentions | Removed {len(removed_mentions_df)} non-political")
print(sorted(removed_mentions_df['mention'].tolist()))
#
# 3) Curated entity lists (normalized: stripped + lowercased)
top_200_political_mentions = set(political_mentions_df.head(200)['mention'])
top_200_political_hashtags = set(political_hashtags_df.head(200)['hashtag'])
political_entities = sorted({str(e).strip().lower() for e in (top_200_political_mentions | top_200_political_hashtags) if str(e).strip()})
removed_words = set(removed_df['hashtag']) | set(removed_mentions_df['mention'])
non_political_words = sorted({str(w).strip().lower() for w in removed_words if str(w).strip()})
#
# 4) Build the binary lookup dictionaries (entity -> label)
political_dict = {e: 1 for e in political_entities}
non_political_dict = {w: 0 for w in non_political_words}
#
# 5) Report dictionary sizes
print('\n=== Dictionary sizes ===')
print('Political mentions (top-200) :', len(top_200_political_mentions))
print('Political hashtags (top-200) :', len(top_200_political_hashtags))
print('Political dictionary (label=1):', len(political_dict))
print('Non-political dictionary (0)  :', len(non_political_dict))
#
# 6) Verify there are no duplicate entries within each dictionary
pol_raw = [str(e).strip().lower() for e in (list(top_200_political_mentions) + list(top_200_political_hashtags)) if str(e).strip()]
nonpol_raw = [str(w).strip().lower() for w in removed_words if str(w).strip()]
pol_dupes = sorted({t for t in pol_raw if pol_raw.count(t) > 1})
nonpol_dupes = sorted({t for t in nonpol_raw if nonpol_raw.count(t) > 1})
print('\n=== Duplicate check (within each dictionary) ===')
print('Political duplicates     :', len(pol_dupes), pol_dupes if pol_dupes else '(none)')
print('Non-political duplicates :', len(nonpol_dupes), nonpol_dupes if nonpol_dupes else '(none)')
#
# 7) Check overlaps between political and non-political dicts (report only)
overlap = sorted(set(political_dict) & set(non_political_dict))
print('\n=== Overlap check (political vs non-political) ===')
print('Overlapping entities     :', len(overlap))
print('CONFLICTS:', overlap if overlap else 'None - dictionaries are disjoint.')
#
# 8) Build lookup tables (DataFrames) in Entity/Label format
political_label_df = pd.DataFrame({'Entity': list(political_dict.keys()), 'Label': list(political_dict.values())})
non_political_label_df = pd.DataFrame({'Entity': list(non_political_dict.keys()), 'Label': list(non_political_dict.values())})
print('\n=== Political dictionary (label=1) preview ===')
display(political_label_df.head(100))
print('\n=== Non-political dictionary (label=0) preview ===')
display(non_political_label_df.head(100))

Kept 12039 political mentions | Removed 26 non-political
['AnupamPKher', 'BBCWorld', 'BCCI', 'CNN', 'DeepSandhu_K', 'ESPNcricinfo', 'ICC', 'IamSanjuSamson', 'Indian_Analyzer', 'OxfordWords', 'Tractor2twitr', 'Twitter', 'WithPGV', '_YogendraYadav', 'akshaykumar', 'ashutosh83B', 'babarazam258', 'bcci', 'elonmusk', 'iamsrk', 'imVkohli', 'khushsundar', 'rihanna', 'twitter', 'vikrantgupta73', 'vivekoberoi']

=== Dictionary sizes ===
Political mentions (top-200) : 200
Political hashtags (top-200) : 200
Political dictionary (label=1): 392
Non-political dictionary (0)  : 69

=== Duplicate check (within each dictionary) ===
Political duplicates     : 7 ['amitshah', 'arvindkejriwal', 'bjp4india', 'incindia', 'narendramodi', 'priyankagandhi', 'rahulgandhi']
Non-political duplicates : 3 ['bcci', 'icc', 'twitter']

=== Overlap check (political vs non-political) ===
Overlapping entities     : 2
CONFLICTS: ['bcci', 'twitter']

=== Political dictionary (label=1) preview ===


,Entity,Label
0,2019elections,1
1,_pallavighosh,1
2,aajtak,1
3,aamaadmiparty,1
4,aap,1
5,aayegatohmodihi,1
6,aayegatomodihi,1
7,abhisar_sharma,1
8,abhoganyay,1
9,abkibaarphirmodisarkar,1



=== Non-political dictionary (label=0) preview ===


,Entity,Label
0,_yogendrayadav,0
1,africa,0
2,akshaykumar,0
3,anupampkher,0
4,ashutosh83b,0
5,babarazam,0
6,babarazam258,0
7,bbcworld,0
8,bcci,0
9,bisleri,0


In [ ]:
# ============================================================
# STEP 5 - Entity-level binary labels, HASHTAGS and MENTIONS SEPARATELY
#   political -> 1 (kept)   |   non-political -> 0 (removed via non_political)
#   entity-level labels only (not tweet labels)
# ============================================================
import pandas as pd
#
# --- HASHTAGS: top-200 political (1) + removed non-political (0) ---
top_200_political_hashtags = set(political_hashtags_df.head(200)['hashtag'])
removed_hashtag_words = set(removed_df['hashtag'])
pol_hashtags = sorted({str(h).strip().lower() for h in top_200_political_hashtags if str(h).strip()})
nonpol_hashtags = sorted({str(h).strip().lower() for h in removed_hashtag_words if str(h).strip()})
hashtag_political_dict = {h: 1 for h in pol_hashtags}
hashtag_non_political_dict = {h: 0 for h in nonpol_hashtags}
#
# --- MENTIONS: top-200 political (1) + removed non-political (0) ---
top_200_political_mentions = set(political_mentions_df.head(200)['mention'])
removed_mention_words = set(removed_mentions_df['mention'])
pol_mentions = sorted({str(m).strip().lower() for m in top_200_political_mentions if str(m).strip()})
nonpol_mentions = sorted({str(m).strip().lower() for m in removed_mention_words if str(m).strip()})
mention_political_dict = {m: 1 for m in pol_mentions}
mention_non_political_dict = {m: 0 for m in nonpol_mentions}
#
# --- Sizes ---
print('=== Dictionary sizes (separate) ===')
print('Hashtags political (1)   :', len(hashtag_political_dict))
print('Hashtags non-political(0):', len(hashtag_non_political_dict))
print('Mentions political (1)   :', len(mention_political_dict))
print('Mentions non-political(0):', len(mention_non_political_dict))
#
# --- Duplicate checks (within each dict) ---
h_pol_raw = [str(h).strip().lower() for h in political_hashtags_df.head(200)['hashtag'] if str(h).strip()]
m_pol_raw = [str(m).strip().lower() for m in political_mentions_df.head(200)['mention'] if str(m).strip()]
h_pol_dupes = sorted({t for t in h_pol_raw if h_pol_raw.count(t) > 1})
m_pol_dupes = sorted({t for t in m_pol_raw if m_pol_raw.count(t) > 1})
print('\n=== Duplicate check ===')
print('Hashtag political duplicates :', len(h_pol_dupes), h_pol_dupes if h_pol_dupes else '(none)')
print('Mention political duplicates :', len(m_pol_dupes), m_pol_dupes if m_pol_dupes else '(none)')
#
# --- Overlap checks (political vs non-political), per entity type, report only ---
hashtag_overlap = sorted(set(hashtag_political_dict) & set(hashtag_non_political_dict))
mention_overlap = sorted(set(mention_political_dict) & set(mention_non_political_dict))
print('\n=== Overlap check (political vs non-political) ===')
print('Hashtag overlaps :', len(hashtag_overlap), hashtag_overlap if hashtag_overlap else 'None - disjoint')
print('Mention overlaps :', len(mention_overlap), mention_overlap if mention_overlap else 'None - disjoint')
#
# --- Lookup tables (Entity/Label) per type ---
hashtag_label_df = pd.DataFrame([(h, 1) for h in pol_hashtags] + [(h, 0) for h in nonpol_hashtags], columns=['Entity', 'Label'])
mention_label_df = pd.DataFrame([(m, 1) for m in pol_mentions] + [(m, 0) for m in nonpol_mentions], columns=['Entity', 'Label'])
print('\n=== Hashtag labels (political=1 then non-political=0) ===')
display(hashtag_label_df.head(10))
display(hashtag_label_df.tail(10))
print('\n=== Mention labels (political=1 then non-political=0) ===')
display(mention_label_df.head(10))
display(mention_label_df.tail(10))

=== Dictionary sizes (separate) ===
Hashtags political (1)   : 200
Hashtags non-political(0): 46
Mentions political (1)   : 199
Mentions non-political(0): 24

=== Duplicate check ===
Hashtag political duplicates : 0 (none)
Mention political duplicates : 1 ['narendramodi']

=== Overlap check (political vs non-political) ===
Hashtag overlaps : 0 None - disjoint
Mention overlaps : 0 None - disjoint

=== Hashtag labels (political=1 then non-political=0) ===


,Entity,Label
0,2019elections,1
1,aap,1
2,aayegatohmodihi,1
3,aayegatomodihi,1
4,abhoganyay,1
5,abkibaarphirmodisarkar,1
6,amethi,1
7,amitshah,1
8,arbitrage,1
9,arvindkejriwal,1


,Entity,Label
236,thiruvananthapuram,0
237,uae,0
238,ukrainerussiawar,0
239,usa,0
240,vaccines,0
241,viratkohli,0
242,windpower,0
243,घर_व,0
244,ज,0
245,म,0



=== Mention labels (political=1 then non-political=0) ===


,Entity,Label
0,_pallavighosh,1
1,aajtak,1
2,aamaadmiparty,1
3,abhisar_sharma,1
4,abpnewshindi,1
5,abpnewstv,1
6,adgpi,1
7,ahmedpatel,1
8,aitcofficial,1
9,akashbanerjee,1


,Entity,Label
213,imvkohli,0
214,indian_analyzer,0
215,khushsundar,0
216,oxfordwords,0
217,rihanna,0
218,tractor2twitr,0
219,twitter,0
220,vikrantgupta73,0
221,vivekoberoi,0
222,withpgv,0


In [ ]:
political_mentions_df['Label'] = ""
political_hashtags_df['Label'] = ""

political_mentions_df.head()
political_hashtags_df.head()

,hashtag,frequency,Label
0,loksabhaelections2019,6634,
1,rahulgandhi,5507,
2,bjp,4373,
3,narendramodi,1990,
4,farmersprotest,1864,


In [ ]:
top_200_mentions = {str(m).strip().lower() for m in political_mentions_df.head(200)['mention']}
top_200_hashtags = {str(h).strip().lower() for h in political_hashtags_df.head(200)['hashtag']}
df['mention_filtered'] = df['mention'].apply(lambda names: [m.lower() for m in names if m.lower() in top_200_mentions])
df['hashtags_filtered'] = df['hashtags'].apply(lambda tags: [t.lower() for t in tags if t.lower() in top_200_hashtags])
political_mentions = set(mention_political_dict.keys())
political_hashtags = set(hashtag_political_dict.keys())
non_political_mentions = set(mention_non_political_dict.keys())
non_political_hashtags = set(hashtag_non_political_dict.keys())
df['has_political_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & political_mentions))
df['has_political_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & political_hashtags))
df['has_nonpolitical_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & non_political_mentions))
df['has_nonpolitical_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & non_political_hashtags))
df['tweet_label'] = (df['has_political_mention'] | df['has_political_hashtag']).astype(int)
total_tweets = len(df)
n_political = df['tweet_label'].sum()
n_nonpolitical = total_tweets - n_political
print(f"Total tweets                                : {total_tweets:,}")
print(f"Tweets labeled political (tweet_label=1)    : {n_political:,} ({n_political/total_tweets*100:.2f}%)")
print(f"Tweets labeled non-political (tweet_label=0): {n_nonpolitical:,} ({n_nonpolitical/total_tweets*100:.2f}%)")
df[['tweet', 'mention', 'hashtags', 'tweet_label']].head(100)


Total tweets                                : 47,990
Tweets labeled political (tweet_label=1)    : 26,314 (54.83%)
Tweets labeled non-political (tweet_label=0): 21,676 (45.17%)


,tweet,mention,hashtags,tweet_label
0,This cartoon by Alok @caricatured speak about the reality of deals in Indian Politics - Backdoor - or Unholy nexus - when all guys unite on controlling money minting Cricket Administration 😎,[caricatured],[],0
1,I am going to post one #BernieAtTheFarmersProtest every day until either the Farmers protest ends or @BernieSanders @SenSanders notices this and highlights the issue. Or I get blocked by Twitter. #Berniememes #BernieSandersMittens #BernieSanders #FarmersProtest 34 https://t.co/2NFSWxE6fA,"[BernieSanders, SenSanders]","[bernieatthefarmersprotest, berniememes, berniesandersmittens, berniesanders, farmersprotest]",1
2,@sardesairajdeep @OfficialUrmila @RahulGandhi So bashing Islamic terrorism is communal but when someone talks about Hindu violence its crappy. Bloody hypocrites. 😪 \n#ModiLaoDeshBanao #ModiHainTohMumkinHain #LokSabhaElections2019 #CongressMuktBharat,"[sardesairajdeep, OfficialUrmila, RahulGandhi]","[modilaodeshbanao, modihaintohmumkinhain, loksabhaelections2019, congressmuktbharat]",1
3,Oooh... that`s right by the zoo... think... in 2 months` time that could be our regular other meeting place,[],[],0
4,@vivekoberoi @narendramodi @OmungKumar @sureshoberoi @sandip_Ssingh @TSeries @anandpandit63 Absolutely very nice and fantastic Sir 😐❤ Your performance very nice🙋‍♂ Because I am see your movie trailer amazing and fantastic ❤🙋‍♂🙏 #LokSabhaElections2019 #AbkiBaarPhirModiSarkar 🙏,"[vivekoberoi, narendramodi, OmungKumar, sureshoberoi, sandip_Ssingh, TSeries, anandpandit63]","[loksabhaelections2019, abkibaarphirmodisarkar]",1
5,_louise Lucky me. There are mystery ingredients as well,[],[],0
6,డిల్లీ అహంకారంపై .. ఆంధ్రుడి పోరాటం అండగా ఉందాం #narendramodi and #ncbn might have political or personal differences But Modi should not undermine #AndhraPradesh people interests and punish them by stopping project funding #DharmaPorataDeeksha #APDemandsJustice #BeWithBabu,[],"[narendramodi, ncbn, andhrapradesh, dharmaporatadeeksha, apdemandsjustice, bewithbabu]",1
7,Dear @RahulGandhi #RahulGandhi 1 side u offer #pyaarkirajnithi on d other hand u wound an entire community repeatedly #SikhGenocide shows yours n yours party's ethics n committment towards aam jantha.. #NYAYforSikhs #Shame #Congress #HuaTohHua #SamInsultsSikhs,[RahulGandhi],"[rahulgandhi, pyaarkirajnithi, sikhgenocide, nyayforsikhs, shame, congress, huatohhua, saminsultssikhs]",1
8,God Bless Ukraine. “The shameful criminalization of freedom of expression must stop.” ~ Marie Struthers https://t.co/j1N1yZ8Sen https://t.co/JDx42B6CgZ #UkraineRussiaWar #Russia #USA #Europe #NATO #China #India #Africa #quote #freedom #freespeech,[],"[ukrainerussiawar, russia, usa, europe, nato, china, india, africa, quote, freedom, freespeech]",0
9,"2/2 Moreover, @kharge has been a tremendous Social reformer, a Great Parliamentarian with a Strong belief in the Constitution of India and Indian secularism ideology with dignity politics 🇮🇳, a loyal warrior of the Congress party @INCIndia @RahulGandhi @priyankagandhi https://t.co/VyJTvNevUf","[kharge, INCIndia, RahulGandhi, priyankagandhi]",[],1


In [ ]:
top_300_mentions = set(political_mentions_df.head(300)['mention'])
top_300_hashtags = set(political_hashtags_df.head(300)['hashtag'])
df['mention_filtered'] = df['mention'].apply(lambda names: [m for m in names if m.lower() in top_200_mentions])
df['hashtags_filtered'] = df['hashtags'].apply(lambda tags: [t for t in tags if t.lower() in top_200_hashtags])
political_mentions = set(mention_political_dict.keys())
political_hashtags = set(hashtag_political_dict.keys())
non_political_mentions = set(mention_non_political_dict.keys())
non_political_hashtags = set(hashtag_non_political_dict.keys())
df['has_political_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & political_mentions))
df['has_political_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & political_hashtags))
df['has_nonpolitical_mention'] = df['mention_filtered'].apply(lambda names: bool(set(names) & non_political_mentions))
df['has_nonpolitical_hashtag'] = df['hashtags_filtered'].apply(lambda tags: bool(set(tags) & non_political_hashtags))
df['tweet_label'] = (df['has_political_mention'] | df['has_political_hashtag']).astype(int)
total_tweets = len(df)
n_political = df['tweet_label'].sum()
n_nonpolitical = total_tweets - n_political
print(f"Total tweets                                : {total_tweets:,}")
print(f"Tweets labeled political (tweet_label=1)    : {n_political:,} ({n_political/total_tweets*100:.2f}%)")
print(f"Tweets labeled non-political (tweet_label=0): {n_nonpolitical:,} ({n_nonpolitical/total_tweets*100:.2f}%)")
df[['tweet', 'mention', 'hashtags', 'tweet_label']].head(100)


Total tweets                                : 47,990
Tweets labeled political (tweet_label=1)    : 25,378 (52.88%)
Tweets labeled non-political (tweet_label=0): 22,612 (47.12%)


,tweet,mention,hashtags,tweet_label
0,This cartoon by Alok @caricatured speak about the reality of deals in Indian Politics - Backdoor - or Unholy nexus - when all guys unite on controlling money minting Cricket Administration 😎,[caricatured],[],0
1,I am going to post one #BernieAtTheFarmersProtest every day until either the Farmers protest ends or @BernieSanders @SenSanders notices this and highlights the issue. Or I get blocked by Twitter. #Berniememes #BernieSandersMittens #BernieSanders #FarmersProtest 34 https://t.co/2NFSWxE6fA,"[BernieSanders, SenSanders]","[bernieatthefarmersprotest, berniememes, berniesandersmittens, berniesanders, farmersprotest]",1
2,@sardesairajdeep @OfficialUrmila @RahulGandhi So bashing Islamic terrorism is communal but when someone talks about Hindu violence its crappy. Bloody hypocrites. 😪 \n#ModiLaoDeshBanao #ModiHainTohMumkinHain #LokSabhaElections2019 #CongressMuktBharat,"[sardesairajdeep, OfficialUrmila, RahulGandhi]","[modilaodeshbanao, modihaintohmumkinhain, loksabhaelections2019, congressmuktbharat]",1
3,Oooh... that`s right by the zoo... think... in 2 months` time that could be our regular other meeting place,[],[],0
4,@vivekoberoi @narendramodi @OmungKumar @sureshoberoi @sandip_Ssingh @TSeries @anandpandit63 Absolutely very nice and fantastic Sir 😐❤ Your performance very nice🙋‍♂ Because I am see your movie trailer amazing and fantastic ❤🙋‍♂🙏 #LokSabhaElections2019 #AbkiBaarPhirModiSarkar 🙏,"[vivekoberoi, narendramodi, OmungKumar, sureshoberoi, sandip_Ssingh, TSeries, anandpandit63]","[loksabhaelections2019, abkibaarphirmodisarkar]",1
5,_louise Lucky me. There are mystery ingredients as well,[],[],0
6,డిల్లీ అహంకారంపై .. ఆంధ్రుడి పోరాటం అండగా ఉందాం #narendramodi and #ncbn might have political or personal differences But Modi should not undermine #AndhraPradesh people interests and punish them by stopping project funding #DharmaPorataDeeksha #APDemandsJustice #BeWithBabu,[],"[narendramodi, ncbn, andhrapradesh, dharmaporatadeeksha, apdemandsjustice, bewithbabu]",1
7,Dear @RahulGandhi #RahulGandhi 1 side u offer #pyaarkirajnithi on d other hand u wound an entire community repeatedly #SikhGenocide shows yours n yours party's ethics n committment towards aam jantha.. #NYAYforSikhs #Shame #Congress #HuaTohHua #SamInsultsSikhs,[RahulGandhi],"[rahulgandhi, pyaarkirajnithi, sikhgenocide, nyayforsikhs, shame, congress, huatohhua, saminsultssikhs]",1
8,God Bless Ukraine. “The shameful criminalization of freedom of expression must stop.” ~ Marie Struthers https://t.co/j1N1yZ8Sen https://t.co/JDx42B6CgZ #UkraineRussiaWar #Russia #USA #Europe #NATO #China #India #Africa #quote #freedom #freespeech,[],"[ukrainerussiawar, russia, usa, europe, nato, china, india, africa, quote, freedom, freespeech]",0
9,"2/2 Moreover, @kharge has been a tremendous Social reformer, a Great Parliamentarian with a Strong belief in the Constitution of India and Indian secularism ideology with dignity politics 🇮🇳, a loyal warrior of the Congress party @INCIndia @RahulGandhi @priyankagandhi https://t.co/VyJTvNevUf","[kharge, INCIndia, RahulGandhi, priyankagandhi]",[],1


In [ ]:
print(political_mentions_df.columns.tolist())
print(political_hashtags_df.columns.tolist())

['mention', 'frequency', 'Label']
['hashtag', 'frequency', 'Label']


In [ ]:
no_mention = df['mention'].apply(lambda x: len(x) == 0)
no_hashtag = df['hashtags'].apply(lambda x: len(x) == 0)
no_both = no_mention & no_hashtag
count_empty = int(no_both.sum())
total = len(df)
print(f" no mention and hashtags= {count_empty:,}/{total:,}")
print(f"% :  {count_empty/total*100:.2f}%")


 no mention and hashtags= 15,248/47,990
% :  31.77%


In [ ]:
print(df.columns.tolist())

['tweet', 'user', 'mention', 'hashtags', 'has_mention', 'has_hashtag', 'clean_text', 'political_score', 'is_political', 'political_mention_hits', 'political_hashtag_hits', 'political_score_mh', 'has_political_signal_mh', 'has_top300_political_hashtag', 'has_top200_political_hashtag', 'mention_filtered', 'hashtags_filtered', 'has_political_mention', 'has_political_hashtag', 'has_nonpolitical_mention', 'has_nonpolitical_hashtag', 'tweet_label']


In [ ]:
df = df[['tweet','clean_text', 'user', 'mention', 'hashtags', 'tweet_label']]

In [ ]:
df.columns

Index(['tweet', 'clean_text', 'user', 'mention', 'hashtags', 'tweet_label'], dtype='object')

In [ ]:
type(df.iloc[0].hashtags)

list

In [ ]:
~df['hashtags'].astype(bool)

,hashtags
0,True
1,False
2,False
3,True
4,False
...,...
47985,True
47986,False
47987,False
47988,True


In [ ]:
len(df[(~df['mention'].astype(bool))&(~df['hashtags'].astype(bool))])

15248

In [ ]:
df[(~df['mention'].astype(bool))&(~df['hashtags'].astype(bool))]

,tweet,clean_text,user,mention,hashtags,tweet_label
3,Oooh... that`s right by the zoo... think... in 2 months` time that could be our regular other meeting place,oooh thats right zoo think months time could regular meeting place,user_3,[],[],0
5,_louise Lucky me. There are mystery ingredients as well,louise lucky mystery ingredients well,user_5,[],[],0
13,yeah I need a hug...cuz I am sick..,yeah need hugcuz sick,user_13,[],[],0
16,So my lucky jade nacklace/matching earrings ain`t so lucky. Lost an earring. Now the chain broke on pendant,lucky jade nacklacematching earrings aint lucky lost earring chain broke pendant,user_17,[],[],0
17,Grabbing coffee from then making mom breakfast,grabbing coffee making mom breakfast,user_18,[],[],0
...,...,...,...,...,...,...
47968,Now standing because my tailbone is killing me,standing tailbone killing,user_50146,[],[],0
47969,_fr Yes I saw the Village but the restaurant in the Village Square has a sign above it that reads 'Digestif.',yes saw village restaurant village square sign reads digestif,user_50147,[],[],0
47979,re: the job ... still waiting my friend. Thanks for asking * I just need a little ... ;) * ? http://blip.fm/~5jehr,job still waiting friend thanks asking need little,user_50157,[],[],0
47985,"_joyner And he can`t even tell me. Me and him are **** done, professionally. ****` ****.",joyner cant even tell done professionally,user_50163,[],[],0


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_text(text):
    return ' '.join(stemmer.stem(word) for word in str(text).split())

df['stemmed_text'] = df['tweet'].apply(stem_text)

In [ ]:
clean_vocab_list = [w for w, s, d in political_vocab[:700]]

tfidf_pol = TfidfVectorizer(
    vocabulary    = clean_vocab_list,  # restrict to cleaned vocab only
    sublinear_tf  = True
)

X_pol_only    = tfidf_pol.fit_transform(df['clean_text'])
political_cols = list(range(X_pol_only.shape[1]))   # all cols = political cols

political_score = np.asarray(X_pol_only.sum(axis=1)).flatten()
df['political_score'] = political_score

non_zero  = political_score[political_score > 0]
threshold = float(np.median(non_zero))
df['is_political'] = (df['political_score'] > 0).astype(int)     # <-- the 0/1 label assignment

pol_count    = int(df['is_political'].sum())
nonpol_count = int((df['is_political']==0).sum())

print(f"Vocabulary used              : {len(clean_vocab_list)}")
print(f"Identified as POLITICAL      : {pol_count:,}  ({pol_count/len(df)*100:.1f}%)")
print(f"Identified as NON-POLITICAL  : {nonpol_count:,}  ({nonpol_count/len(df)*100:.1f}%)")

Vocabulary used              : 675
Identified as POLITICAL      : 44,336  (92.4%)
Identified as NON-POLITICAL  : 3,654  (7.6%)


In [ ]:
# STEP 1: TF-IDF on Full Dataset
# Independent baseline - no labeling yet
# Goal: Apply TF-IDF vectorization to show score distribution

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Apply TF-IDF on full dataset
tfidf = TfidfVectorizer(max_features=1000, min_df=2, max_df=0.8, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Total tweets: {tfidf_matrix.shape[0]}")
print(f"Features (vocabulary size): {tfidf_matrix.shape[1]}")
print(f"Sparsity: {1.0 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))*100:.2f}%")

TF-IDF Matrix shape: (47990, 1000)
Total tweets: 47990
Features (vocabulary size): 1000
Sparsity: 0.33%


In [ ]:
# CELL 55 (move this BEFORE cell 53) — define political_vocab_set first
political_vocab_set = {w.lower() for w, s, d in political_vocab}

# Then CELL 53 — REPLACE with TF-IDF vocab included in NER validation

import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])

RELEVANT_ENTITY_LABELS = {"PERSON", "ORG", "GPE", "NORP"}

# Combined knowledge base — all three sources
political_entities_combined = set()
political_entities_combined.update({m.lower() for m in political_mentions})        # mentions
political_entities_combined.update({h.lower() for h in political_hashtags})        # hashtags
political_entities_combined.update(political_vocab_set)                             # TF-IDF vocab ← NEW

# Normalize: remove spaces, lowercase
political_entities_combined = {e.replace(" ", "").lower() for e in political_entities_combined}

# Only run NER on tweets that mention/hashtag did NOT label
no_mention = df['mention'].apply(len) == 0
no_hashtag = df['hashtags'].apply(len) == 0
no_mh_signal = (df['tweet_label'] == 0)          # ← KEY: only process unlabeled tweets

ner_idx   = df.loc[no_mh_signal].index
ner_texts = df.loc[no_mh_signal, 'tweet'].astype(str).tolist()
print(f"Tweets entering NER : {len(ner_texts):,} (only those without mention/hashtag signal)")

entities_list, entity_counts, political_matches_list, labels = [], [], [], []

for doc in nlp.pipe(ner_texts, batch_size=200):
    entities = [ent.text for ent in doc.ents if ent.label_ in RELEVANT_ENTITY_LABELS]

    # Normalize entities for matching
    def normalize(e):
        return e.lower().replace(" ", "").strip()

    political_matches = sum(
        1 for e in entities
        if normalize(e) in political_entities_combined
    )
    label = 1 if political_matches > 0 else 0

    entities_list.append(entities)
    entity_counts.append(len(entities))
    political_matches_list.append(political_matches)
    labels.append(label)

# Initialize all to 0
df['ner_entities']          = [[] for _ in range(len(df))]
df['ner_entity_count']      = 0
df['ner_political_matches'] = 0
df['ner_label']             = 0

# Write only to NER-processed rows
df.loc[ner_idx, 'ner_entities']          = pd.Series(entities_list,         index=ner_idx, dtype=object)
df.loc[ner_idx, 'ner_entity_count']      = pd.Series(entity_counts,         index=ner_idx)
df.loc[ner_idx, 'ner_political_matches'] = pd.Series(political_matches_list, index=ner_idx)
df.loc[ner_idx, 'ner_label']             = pd.Series(labels,                 index=ner_idx)

n_pol   = int(df.loc[ner_idx, 'ner_label'].sum())
n_total = len(ner_idx)
print(f"NER labeled Political    : {n_pol:,}  ({n_pol/n_total*100:.2f}%)")
print(f"NER labeled Non-Political: {n_total-n_pol:,}")

Tweets entering NER : 22,612 (only those without mention/hashtag signal)
NER labeled Political    : 7,754  (34.29%)
NER labeled Non-Political: 14,858


In [ ]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
RELEVANT_ENTITY_LABELS = {"PERSON", "ORG", "GPE", "NORP"}

# ===== BUILD THE COMBINED KNOWLEDGE BASE =====
# Source 1: Top 200 political mentions
political_mentions_set = set({x.lower() for x in political_mentions})

# Source 2: Top 200 political hashtags
political_hashtags_set = set({h.lower() for h in political_hashtags})

# Source 3: Top 800 TF-IDF political vocabulary
political_vocab_set = {w.lower() for w, s, d in political_vocab[:800]}

# COMBINE ALL THREE
political_entities_combined = set()
political_entities_combined.update(political_mentions_set)
political_entities_combined.update(political_hashtags_set)
political_entities_combined.update(political_vocab_set)

# Normalize: remove spaces, lowercase
political_entities_combined = {e.replace(" ", "").lower() for e in political_entities_combined}

print(f"Combined knowledge base size:")
print(f"  Top 200 mentions: {len(political_mentions_set)}")
print(f"  Top 200 hashtags: {len(political_hashtags_set)}")
print(f"  Top 800 TF-IDF vocab: {len(political_vocab_set)}")
print(f"  Total (combined): {len(political_entities_combined)}")

# ===== ONLY PROCESS TWEETS WITHOUT MENTION/HASHTAG SIGNAL =====
no_mention = df['mention'].apply(len) == 0
no_hashtag = df['hashtags'].apply(len) == 0
remaining_mask = no_mention & no_hashtag

if 'tfidf_label' in df.columns:
    remaining_mask = remaining_mask & (df['tfidf_label'] == 0)

ner_idx = df.loc[remaining_mask].index
ner_texts = df.loc[remaining_mask, 'tweet'].astype(str).tolist()
print(f"\nTweets entering NER: {len(ner_texts):,} / {len(df):,}")

# ===== BATCHED NER PROCESSING =====
entities_list, entity_counts, political_matches_list, labels = [], [], [], []

for doc in nlp.pipe(ner_texts, batch_size=200):
    entities = [ent.text for ent in doc.ents if ent.label_ in RELEVANT_ENTITY_LABELS]
    entity_count = len(entities)

    # Normalize entities and validate against COMBINED knowledge base
    political_matches = sum(
        1 for e in entities
        if e.lower().replace(" ", "").strip() in political_entities_combined
    )
    label = 1 if (entity_count >= 3 and political_matches > 0) else 0

    entities_list.append(entities)
    entity_counts.append(entity_count)
    political_matches_list.append(political_matches)
    labels.append(label)

# ===== POPULATE RESULTS =====
df['ner_entities'] = [[]] * len(df)
df['ner_entity_count'] = 0
df['ner_political_matches'] = 0
df['ner_label'] = 0

df.loc[ner_idx, 'ner_entities'] = pd.Series(entities_list, index=ner_idx, dtype=object)
df.loc[ner_idx, 'ner_entity_count'] = pd.Series(entity_counts, index=ner_idx)
df.loc[ner_idx, 'ner_political_matches'] = pd.Series(political_matches_list, index=ner_idx)
df.loc[ner_idx, 'ner_label'] = pd.Series(labels, index=ner_idx)

# ===== REPORT RESULTS =====
n_political = int(df.loc[ner_idx, 'ner_label'].sum())
n_total = len(ner_idx)

print("\n=== NER LABELING RESULTS ===")
print(f"Tweets processed by NER: {n_total:,}")
print(f"Labelled POLITICAL (ner_label=1): {n_political:,} ({n_political/n_total*100:.2f}%)")
print(f"Labelled NON-POLITICAL (ner_label=0): {n_total-n_political:,} ({(n_total-n_political)/n_total*100:.2f}%)")

ner_results = df.loc[ner_idx, [
    'tweet', 'mention', 'hashtags',
    'ner_entities', 'ner_entity_count', 'ner_political_matches', 'ner_label'
]]

print("\nSample POLITICAL (ner_label=1):")
display(ner_results[ner_results['ner_label'] == 1].head(5))

print("\nSample NON-POLITICAL (ner_label=0):")
display(ner_results[ner_results['ner_label'] == 0].head(5))

Combined knowledge base size:
  Top 200 mentions: 199
  Top 200 hashtags: 200
  Top 800 TF-IDF vocab: 675
  Total (combined): 927

Tweets entering NER: 15,248 / 47,990

=== NER LABELING RESULTS ===
Tweets processed by NER: 15,248
Labelled POLITICAL (ner_label=1): 797 (5.23%)
Labelled NON-POLITICAL (ner_label=0): 14,451 (94.77%)

Sample POLITICAL (ner_label=1):


,tweet,mention,hashtags,ner_entities,ner_entity_count,ner_political_matches,ner_label
98,"It's a chaotic pivotal time in global politics and global economics with Russia, China, Brazil, UK, Italy, US all realigning. Modi seems more obsessed with Gujarat elections &amp; Rahul's beard than actually being the Prime Minister of India. Modi is a pracharak before he is Indian.",[],[],"[Russia, China, Brazil, UK, Italy, US, Gujarat, Rahul, India, Modi, Indian]",11,9,1
114,"Indian historian here we have achieved Thomas Babington Macaulay’s dream in Rishi Sunak, Indian in blood British in taste and education in this case also reactionary in politics.",[],[],"[Indian, Thomas Babington Macaulay’s, Rishi Sunak, Indian, British]",5,4,1
166,BJP doing dirty politics to stop AAP govt from sending teachers to Finland for training: Sisodia- The New Indian Express https://t.co/wD71LQlFD9,[],[],"[BJP, Finland, New Indian]",3,1,1
217,I see lots of positives from Rahul Gandhi's Bharat Jodo Yatra. He has evolved as a new leader &amp; people have started connecting with him. We will see immediate results in the Gujarat polls. Bharat Jodo Yatra has given us RaGa 2.0 which is good news for Indian Politics &amp; Economy.,[],[],"[Rahul Gandhi's, Gujarat, Bharat, Jodo Yatra, Indian Politics &]",5,2,1
326,"‘A Pakistani as London Mayor , an Indian as PM’ but you played Ethnic politics against Ngige in 2010, &amp; BAT got virtually nothing in your Region despite campaigning. You won’t give a nail but you want an Arm in return. Clannish Hypocrite . Peter Obi Yoruba Labour Tinubu Bigot https://t.co/c9Eb76BgNL",[],[],"[Pakistani, London Mayor, Indian, Ngige, BAT, Clannish Hypocrite, Peter Obi, Tinubu Bigot https://t.co/c9Eb76BgNL]",8,2,1



Sample NON-POLITICAL (ner_label=0):


,tweet,mention,hashtags,ner_entities,ner_entity_count,ner_political_matches,ner_label
3,Oooh... that`s right by the zoo... think... in 2 months` time that could be our regular other meeting place,[],[],[],0,0,0
5,_louise Lucky me. There are mystery ingredients as well,[],[],[],0,0,0
13,yeah I need a hug...cuz I am sick..,[],[],[],0,0,0
16,So my lucky jade nacklace/matching earrings ain`t so lucky. Lost an earring. Now the chain broke on pendant,[],[],[],0,0,0
17,Grabbing coffee from then making mom breakfast,[],[],[],0,0,0


In [ ]:
political_matches = sum

n_political = int(df.loc[ner_idx, 'ner_label'].sum())
n_total = len(ner_idx)

print("=== NER LABELING RESULTS ===")
print(f"Tweets processed by NER            : {n_total:,}")
print(f"Labelled POLITICAL (ner_label=1)   : {n_political:,} ({n_political/n_total*100:.2f}%)")
print(f"Labelled NON-POLITICAL (ner_label=0): {n_total-n_political:,} ({(n_total-n_political)/n_total*100:.2f}%)")

ner_results_table = df.loc[ner_idx, [
    'tweet', 'user', 'mention', 'hashtags',
    'ner_entities', 'ner_entity_count', 'ner_political_matches', 'ner_label'
]]

print("\nSample POLITICAL (ner_label=1):")
display(ner_results_table[ner_results_table['ner_label'] == 1].head(10))

print("\nSample NON-POLITICAL (ner_label=0):")
display(ner_results_table[ner_results_table['ner_label'] == 0].head(10))

ner_results_table.to_csv('ner_labeling_results.csv', index=False)

=== NER LABELING RESULTS ===
Tweets processed by NER            : 15,248
Labelled POLITICAL (ner_label=1)   : 797 (5.23%)
Labelled NON-POLITICAL (ner_label=0): 14,451 (94.77%)

Sample POLITICAL (ner_label=1):


,tweet,user,mention,hashtags,ner_entities,ner_entity_count,ner_political_matches,ner_label
98,"It's a chaotic pivotal time in global politics and global economics with Russia, China, Brazil, UK, Italy, US all realigning. Modi seems more obsessed with Gujarat elections &amp; Rahul's beard than actually being the Prime Minister of India. Modi is a pracharak before he is Indian.",user_102,[],[],"[Russia, China, Brazil, UK, Italy, US, Gujarat, Rahul, India, Modi, Indian]",11,9,1
114,"Indian historian here we have achieved Thomas Babington Macaulay’s dream in Rishi Sunak, Indian in blood British in taste and education in this case also reactionary in politics.",user_118,[],[],"[Indian, Thomas Babington Macaulay’s, Rishi Sunak, Indian, British]",5,4,1
166,BJP doing dirty politics to stop AAP govt from sending teachers to Finland for training: Sisodia- The New Indian Express https://t.co/wD71LQlFD9,user_172,[],[],"[BJP, Finland, New Indian]",3,1,1
217,I see lots of positives from Rahul Gandhi's Bharat Jodo Yatra. He has evolved as a new leader &amp; people have started connecting with him. We will see immediate results in the Gujarat polls. Bharat Jodo Yatra has given us RaGa 2.0 which is good news for Indian Politics &amp; Economy.,user_225,[],[],"[Rahul Gandhi's, Gujarat, Bharat, Jodo Yatra, Indian Politics &]",5,2,1
326,"‘A Pakistani as London Mayor , an Indian as PM’ but you played Ethnic politics against Ngige in 2010, &amp; BAT got virtually nothing in your Region despite campaigning. You won’t give a nail but you want an Arm in return. Clannish Hypocrite . Peter Obi Yoruba Labour Tinubu Bigot https://t.co/c9Eb76BgNL",user_342,[],[],"[Pakistani, London Mayor, Indian, Ngige, BAT, Clannish Hypocrite, Peter Obi, Tinubu Bigot https://t.co/c9Eb76BgNL]",8,2,1
366,bhupendrachaub shivsena bjpindia bjp ss share ideolog hypocrisi turn blind tdp alli congress tmc alli congress polit illicit mahathugbhandan,user_384,[],[],"[bhupendrachaub shivsena bjpindia bjp, alli, congress, tmc alli, congress, mahathugbhandan]",6,2,1
375,Indian activist Umar Khalid gets a week’s bail — and a gag order | Politics News – EAST AUTO NEWS https://t.co/ryAzr5oD7v,user_393,[],[],"[Indian, Umar Khalid, Politics News]",3,1,1
383,"Politics of Religion In 2019 elections, 60% Hindu voters who think it is very important to be Hindu and to speak Hindi to be truly Indian cast their vote for the BJP Only 33% among Hindu voters who feel less strongly about both these aspects of national identity voted for BJP.",user_401,[],[],"[Hindu, Hindu, Hindi, Indian, BJP, Hindu, BJP]",7,7,1
424,despite bjp narawa harkatoona the pakistani side practiced self restraint and handled the predicament at hand with caution and maturity zindabad khkuley watan lt 3 indiansurgicaldrama loc pakarmy psl8 peshawar,user_445,[],[],"[narawa, harkatoona the, pakistani, khkuley watan]",4,1,1
433,request bjpindia modi ji amitshah jipleas send spokesperson debat channel anchor abl control congress peopl peopl watch debat listen bjp spokesperson idiot anchor amp congress peopl dangal aarpaar hallabol,user_454,[],[],"[request bjpindia, amitshah jipleas, congress, congress]",4,2,1



Sample NON-POLITICAL (ner_label=0):


,tweet,user,mention,hashtags,ner_entities,ner_entity_count,ner_political_matches,ner_label
3,Oooh... that`s right by the zoo... think... in 2 months` time that could be our regular other meeting place,user_3,[],[],[],0,0,0
5,_louise Lucky me. There are mystery ingredients as well,user_5,[],[],[],0,0,0
13,yeah I need a hug...cuz I am sick..,user_13,[],[],[],0,0,0
16,So my lucky jade nacklace/matching earrings ain`t so lucky. Lost an earring. Now the chain broke on pendant,user_17,[],[],[],0,0,0
17,Grabbing coffee from then making mom breakfast,user_18,[],[],[],0,0,0
18,hey Padster...it`s a dirt track. thx for the info! I got 3 miles in,user_20,[],[],[],0,0,0
20,Or a Mexican wrestler. They year capes too. You probably haven`t the build for it though.,user_22,[],[],[Mexican],1,0,0
30,"I am just `okay-okay` .. like the rest of the sane population in the world, I hate mondays",user_33,[],[],[],0,0,0
31,"gotta buy the second season of ghost whisperer now, but noo moneyy",user_34,[],[],[noo moneyy],1,0,0
36,Take that back on the cast...one dropped last night!,user_39,[],[],[],0,0,0


In [ ]:
# Define political_vocab_set from TF-IDF vocabulary
# Convert the TF-IDF vocabulary (array of words) into a set for fast lookup
political_vocab_set = set(vocab)

In [ ]:
# STEP 3: NMF Topic Modeling on Filtered Dataset
# Unsupervised topic extraction with 5 topics
# User will manually inspect and assign political/non-political labels

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import NMF
import pandas as pd

print("\nSTEP 3: NMF Topic Modeling")
print("=" * 80)

# --- FIX 1: df_remaining must be defined from the tweets still
# unlabelled after mention/hashtag + TF-IDF + NER. Recompute it here
# directly so this cell doesn't silently depend on an earlier cell
# that may not have run (this was the root cause of df_remaining being
# undefined before) ---
no_mention = df['mention'].apply(len) == 0
no_hashtag = df['hashtags'].apply(len) == 0
remaining_mask = no_mention & no_hashtag

if 'tfidf_label' in df.columns:
    remaining_mask = remaining_mask & (df['tfidf_label'] == 0)
if 'ner_label' in df.columns:
    remaining_mask = remaining_mask & (df['ner_label'] == 0)

df_remaining = df.loc[remaining_mask].copy()
print(f"Tweets entering NMF: {len(df_remaining):,}")

# Create CountVectorizer for NMF (better for topic modeling than TF-IDF)
count_vectorizer = CountVectorizer(
    max_features=2000,
    min_df=2,
    max_df=0.8,
    stop_words='english',
    ngram_range=(1, 2)  # unigrams and bigrams
)

# Fit on filtered dataset
count_matrix = count_vectorizer.fit_transform(df_remaining['clean_text'])
print(f"Count matrix shape: {count_matrix.shape}")
print(f"Vocabulary size: {len(count_vectorizer.get_feature_names_out())}")

# Apply NMF with 5 topics
n_topics = 12
print(f"\nTraining NMF model with {n_topics} topics...")
nmf = NMF(n_components=n_topics, random_state=42, max_iter=300, init='nndsvd')
nmf.fit(count_matrix)
print("NMF model trained successfully")

# Display top words for each topic
feature_names = count_vectorizer.get_feature_names_out()

print("\n" + "=" * 80)
print("TOP WORDS FOR EACH TOPIC")
print("=" * 80)

topic_words = {}
for topic_idx, topic in enumerate(nmf.components_):
    top_indices = topic.argsort()[-10:][::-1]  # top 10 words
    top_words = [feature_names[i] for i in top_indices]
    topic_words[topic_idx] = top_words
    print(f"\nTopic {topic_idx}: {', '.join(top_words)}")

# Assign topics to tweets and get topic distribution
W = nmf.transform(count_matrix)  # Document-topic matrix
df_remaining['dominant_topic'] = W.argmax(axis=1)
df_remaining['topic_strength'] = W.max(axis=1)

print("\n" + "=" * 80)
print("TOPIC DISTRIBUTION IN FILTERED DATASET")
print("=" * 80)
print(f"Total tweets analyzed: {len(df_remaining)}")
print("\nTweets per topic:")
print(df_remaining['dominant_topic'].value_counts().sort_index())
print(f"\nAverage topic strength: {df_remaining['topic_strength'].mean():.3f}")
print("\n*** NEXT STEP: Manually inspect topics above and assign political/non-political labels ***")
print("*** Edit topic_label below, then re-run this cell to apply it ***")

# --- FIX 2: removed the stray trailing ')' that caused the SyntaxError
# and made everything above unreachable ---

# fill this in after reading the topic words printed above
# e.g. topic_label = {0: 1, 1: 0, 2: 1, 3: 0, 4: 0}   # 1=political, 0=non-political
topic_label = {}

if topic_label:
    df_remaining['nmf_label'] = df_remaining['dominant_topic'].map(topic_label)

    # --- FIX 3: same safe pd.Series(..., index=...) pattern as the NER fix,
    # to avoid ragged/misaligned assignment back into the full df ---
    df['nmf_label'] = 0
    df.loc[df_remaining.index, 'nmf_label'] = pd.Series(
        df_remaining['nmf_label'].values, index=df_remaining.index
    )

    n_political = int(df.loc[df_remaining.index, 'nmf_label'].sum())
    print(f"\nLabelled POLITICAL (nmf_label=1)   : {n_political:,} ({n_political/len(df_remaining)*100:.2f}%)")
    print(f"Labelled NON-POLITICAL (nmf_label=0): {len(df_remaining)-n_political:,}")
else:
    print("\n⚠️ topic_label is empty — fill it in based on the printed topic words, then re-run this cell.")


STEP 3: NMF Topic Modeling
Tweets entering NMF: 14,451
Count matrix shape: (14451, 2000)
Vocabulary size: 2000

Training NMF model with 12 topics...
NMF model trained successfully

TOP WORDS FOR EACH TOPIC

Topic 0: day, mothers, happy, mothers day, happy mothers, moms, mom, day mothers, day moms, great

Topic 1: politics, indian, indian politics, amp, people, political, new, party, india, politicians

Topic 2: like, feel, looks, looks like, feel like, look, dont like, look like, bad, lol

Topic 3: modi, amp, india, govt, attack, modi govt, pakistan, want, govern, peopl

Topic 4: good, morning, good morning, night, hope, really, day, good night, thats, luck

Topic 5: today, going, home, day, miss, new, tomorrow, hope, tonight, fun

Topic 6: dont, know, really, think, want, dont know, feel, lol, dont think, dont want

Topic 7: love, lol, really, new, thanks, great, mom, happy, haha, youre

Topic 8: work, tomorrow, want, day, work tomorrow, work work, home, weekend, doesnt, need

Topic 

In [ ]:
topic_label = {
    0: 0,   # Non-political: mothers day, greetings
    1: 1,   # POLITICAL: politics, indian politics, politicians, party
    2: 0,   # Non-political: generic feelings/emotions
    3: 1,   # POLITICAL: modi, govt, pakistan, governance
    4: 0,   # Non-political: greetings, time expressions
    5: 0,   # Non-political: daily routines, personal activities
    6: 0,   # Non-political: generic expressions of uncertainty
    7: 0,   # Non-political: appreciation, affection, greetings
    8: 0,   # Non-political: work, schedules, personal routines
    9: 0,   # Non-political: personal experiences/activities
    10: 0,  # Non-political: time, generic emotions
    11: 1   # POLITICAL: BJP, Congress, voting, nation, Delhi, support
}

In [ ]:
print("\n=== STEP 4: LDA Topic Modeling for Political Tweet Labeling ===")

# Install gensim if needed
import subprocess
subprocess.run(['pip', 'install', '-q', 'gensim'], check=False)

from gensim import corpora, models
from gensim.parsing.preprocessing import STOPWORDS

# Prepare texts for LDA
texts = [tweet.lower().split() for tweet in df_remaining['clean_text']]

# Create dictionary and corpus
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# Train LDA model with 5 topics (same as NMF for comparison)
print("Training LDA model...")
lda_model = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=5, random_state=42, passes=10, alpha='auto', eta='auto')

print("\nLDA Topics discovered:")
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")

    # Extract dominant LDA topic for each tweet
    df_remaining['lda_dominant_topic'] = [max(lda_model.get_document_topics(bow), key=lambda x: x[1])[0] if lda_model.get_document_topics(bow) else 0 for bow in corpus]

    print("\nLDA Topic distribution:")
    print(df_remaining['lda_dominant_topic'].value_counts().sort_index())

    # LDA-based labeling: Topics containing political keywords get label 1
    political_topics = [1, 4]  # These topics seem related to politics based on keywords
    df_remaining['lda_label'] = df_remaining['lda_dominant_topic'].apply(lambda x: 1 if x in political_topics else 0)

    print(f"\nLDA Labeling Results:")
    print(f"Political tweets (LDA): {(df_remaining['lda_label'] == 1).sum()}")
    print(f"Non-Political tweets (LDA): {(df_remaining['lda_label'] == 0).sum()}")
    print(f"LDA Label distribution:\n{df_remaining['lda_label'].value_counts()}")


=== STEP 4: LDA Topic Modeling for Political Tweet Labeling ===
Training LDA model...

LDA Topics discovered:
Topic 0: 0.026*"politics" + 0.024*"indian" + 0.006*"old" + 0.004*"last" + 0.004*"rain" + 0.004*"ago" + 0.004*"found" + 0.004*"people" + 0.003*"years" + 0.003*"year"

LDA Topic distribution:
lda_dominant_topic
0    1664
1    1683
2    6394
3    1238
4    3472
Name: count, dtype: int64

LDA Labeling Results:
Political tweets (LDA): 5155
Non-Political tweets (LDA): 9296
LDA Label distribution:
lda_label
0    9296
1    5155
Name: count, dtype: int64
Topic 1: 0.024*"modi" + 0.016*"bjp" + 0.007*"amp" + 0.006*"india" + 0.004*"govt" + 0.004*"time" + 0.004*"eat" + 0.003*"indian" + 0.003*"vote" + 0.003*"peopl"

LDA Topic distribution:
lda_dominant_topic
0    1664
1    1684
2    6392
3    1238
4    3473
Name: count, dtype: int64

LDA Labeling Results:
Political tweets (LDA): 5157
Non-Political tweets (LDA): 9294
LDA Label distribution:
lda_label
0    9294
1    5157
Name: count, dtype: in

In [ ]:


if 'political_label' not in df.columns:
    df['political_label'] = 0  # 0 = Non-Political, 1 = Political

# =====================================================================
# Display Sample Political and Non-Political Tweets
# =====================================================================

# Filter display columns to only those that exist
display_cols = [c for c in display_cols if c in df.columns]

# Check if we have any political tweets
num_political = (df['political_label'] == 1).sum()
num_non_political = (df['political_label'] == 0).sum()

print(f"\n📊 Label Distribution:")
print(f"   Political tweets: {num_political}")
print(f"   Non-Political tweets: {num_non_political}")
print(f"   Total tweets: {len(df)}")

# Display samples if available
if num_political > 0:
    print("\n✅ Sample POLITICAL tweets:")
    try:
        display(df[df['political_label'] == 1][display_cols].sample(
            min(5, num_political),  # Show up to 5, or fewer if not available
            random_state=1
        ))
    except Exception as e:
        print(f"   Could not display political samples: {e}")
else:
    print("\n⚠️  No political tweets found in the dataset.")

if num_non_political > 0:
    print("\n✅ Sample NON-POLITICAL tweets:")
    try:
        display(df[df['political_label'] == 0][display_cols].sample(
            min(20, num_non_political),  # Show up to 20, or fewer if not available
            random_state=1
        ))
    except Exception as e:
        print(f"   Could not display non-political samples: {e}")
else:
    print("\n⚠️  No non-political tweets found in the dataset.")


📊 Label Distribution:
   Political tweets: 25378
   Non-Political tweets: 22612
   Total tweets: 47990

✅ Sample POLITICAL tweets:


,tweet,ner_entities,lda_dominant_topic,nmf_dominant_topic
37278,"@khanavo @Ajayvchandra1 @vikramchandra Modi looks (or is made to look) bigger than #BJP and BJP bigger than India! Cause for concern Isn’t it? By BJP’s own yard stick, #LKAdvani advice in priority_ Nation, Party and individuals.",[],<NA>,<NA>
28517,@HardikPatel_ @ahmedpatel @SATAVRAJEEV @avinashpandeinc @priyankagandhi @sherryontopp @RahulGandhi @ashokgehlot51 @JM_Scindia @AmitChavdaINC @NewIndianXpress #LokSabhaElections2019 What about patidar andalon???? People will not forget who had lost their life because of your political gain,[],<NA>,<NA>
23293,@sardesairajdeep @RahulGandhi @INCIndia how idiotically the #BJP in opposition is hell bent upon establishing one upmanship,[],<NA>,<NA>
27891,"@sagarikaghose what party politics? if after 7 decades of your akka rule, we did not have war memorial, its only #NarendraModi could do, what is harm speaking truth?\n\nwhen ur boss @RahulGandhi do same #KhoonKiDalali, why u silent?\n\n#IndiaStrikesBack #airstrike #Balakot",[],<NA>,<NA>
41153,Yuzvendra Chahal gets his fourth wicket of the match. #INDvsENG 💓🔥🏏 #yuzvendrachahal #RohitSharma #JosButtler #moeenali #jonnybairstow #davidwilley #England #India #CricbuzzLive #CricketTwitter #Cricket #HardikPandya #MohammedShami #PlayBold #WhistlePodu #champion #BCCI #ICC https://t.co/0EaMoRTUZi,[],<NA>,<NA>



✅ Sample NON-POLITICAL tweets:


,tweet,ner_entities,lda_dominant_topic,nmf_dominant_topic
9401,- whoa. 'sack chasing whores' ? that is soo out of character for you to say that... bring back my sweet Superman,[Superman],3,<NA>
23235,"NeeRain was awarded as an emerging startup in water conservation by EDII, Ahmedabad. Shri. Ashwini Vaishnav, Hon. Minister of Railways, Communications and Electronics &amp; Information Technology, Government of India. #rain #Water #governmentofindia #India #GroundWater #Neerain https://t.co/GVbazE0mGT",[],<NA>,<NA>
34845,"mmm mmm mmm! tss tsss tss. LOL, having way too much fun being bored. i miss him","[mmm mmm mmm, LOL]",3,<NA>
10487,Past experiences of being a redhead (eg. discrimination) or even just general info bout gingers in society! It all helps,[],0,<NA>
17625,"#HappyBirthday to my friend and colleague, Shri @raghav_chadha #youthicon of Indian politics. Your dedication to the cause of building @AamAadmiParty is an inspiration to us all. May God bless you with good health, long life and abundance of happiness. @AAPDelhi @AAPPunjab https://t.co/dXVvp7Y9GR",[],<NA>,<NA>
392,Blogging-- http://13tolife.us/ Mentioning links to contests where you might just win a free book or two!,[],4,<NA>
3975,"lol i know but it was just so funny, ahaha",[],4,<NA>
45648,@Awhadspeaks Kindly keep doing same social services and don't involve in politics. We need leader like Tata and you can follow same.,[],<NA>,<NA>
12424,Indian cricket team promised that they must defeat by any cricket team of world. The selection committee of BCCI now prove that they boast by politics. Nonsense Indian cricket team.,"[Indian, BCCI, Indian]",<NA>,<NA>
41201,hate u.... I have 2 wait one week to see it cuz here (Puerto Rico) is still coming soon....,[Puerto Rico],2,<NA>


In [ ]:
if 'nmf_dominant_topic' not in df.columns:
    df['nmf_dominant_topic'] = pd.NA
if 'lda_dominant_topic' not in df.columns:
    df['lda_dominant_topic'] = pd.NA

if 'df_remaining' in globals():
    if 'nmf_dominant_topic' in df_remaining.columns:
        df.loc[df_remaining.index, 'nmf_dominant_topic'] = df_remaining['nmf_dominant_topic']
    if 'lda_dominant_topic' in df_remaining.columns:
        df.loc[df_remaining.index, 'lda_dominant_topic'] = df_remaining['lda_dominant_topic']

if 'ner_entities' not in df.columns:
    df['ner_entities'] = [[]] * len(df)

display_cols = [
    'tweet',
    'mention_filtered',     # top-200 political mentions found in this tweet
    'hashtags_filtered',    # top-200 political hashtags found in this tweet
    'ner_entities',         # NER-extracted entities
    'lda_dominant_topic',   # LDA topic id (NaN if tweet never reached LDA stage)
    'nmf_dominant_topic',   # NMF topic id (NaN if tweet never reached NMF stage)
    'political_label'
]
display_cols = [c for c in display_cols if c in df.columns]
print("\nSample POLITICAL:")
display(df[df['political_label'] == 1][display_cols].sample(5, random_state=1))

print("\nSample NON-POLITICAL:")
display(df[df['political_label'] == 0][display_cols].sample(20, random_state=1))


Sample POLITICAL:


,tweet,ner_entities,lda_dominant_topic,nmf_dominant_topic,political_label
37278,"@khanavo @Ajayvchandra1 @vikramchandra Modi looks (or is made to look) bigger than #BJP and BJP bigger than India! Cause for concern Isn’t it? By BJP’s own yard stick, #LKAdvani advice in priority_ Nation, Party and individuals.",[],<NA>,<NA>,1
28517,@HardikPatel_ @ahmedpatel @SATAVRAJEEV @avinashpandeinc @priyankagandhi @sherryontopp @RahulGandhi @ashokgehlot51 @JM_Scindia @AmitChavdaINC @NewIndianXpress #LokSabhaElections2019 What about patidar andalon???? People will not forget who had lost their life because of your political gain,[],<NA>,<NA>,1
23293,@sardesairajdeep @RahulGandhi @INCIndia how idiotically the #BJP in opposition is hell bent upon establishing one upmanship,[],<NA>,<NA>,1
27891,"@sagarikaghose what party politics? if after 7 decades of your akka rule, we did not have war memorial, its only #NarendraModi could do, what is harm speaking truth?\n\nwhen ur boss @RahulGandhi do same #KhoonKiDalali, why u silent?\n\n#IndiaStrikesBack #airstrike #Balakot",[],<NA>,<NA>,1
41153,Yuzvendra Chahal gets his fourth wicket of the match. #INDvsENG 💓🔥🏏 #yuzvendrachahal #RohitSharma #JosButtler #moeenali #jonnybairstow #davidwilley #England #India #CricbuzzLive #CricketTwitter #Cricket #HardikPandya #MohammedShami #PlayBold #WhistlePodu #champion #BCCI #ICC https://t.co/0EaMoRTUZi,[],<NA>,<NA>,1



Sample NON-POLITICAL:


,tweet,ner_entities,lda_dominant_topic,nmf_dominant_topic,political_label
9401,- whoa. 'sack chasing whores' ? that is soo out of character for you to say that... bring back my sweet Superman,[Superman],3,<NA>,0
23235,"NeeRain was awarded as an emerging startup in water conservation by EDII, Ahmedabad. Shri. Ashwini Vaishnav, Hon. Minister of Railways, Communications and Electronics &amp; Information Technology, Government of India. #rain #Water #governmentofindia #India #GroundWater #Neerain https://t.co/GVbazE0mGT",[],<NA>,<NA>,0
34845,"mmm mmm mmm! tss tsss tss. LOL, having way too much fun being bored. i miss him","[mmm mmm mmm, LOL]",3,<NA>,0
10487,Past experiences of being a redhead (eg. discrimination) or even just general info bout gingers in society! It all helps,[],0,<NA>,0
17625,"#HappyBirthday to my friend and colleague, Shri @raghav_chadha #youthicon of Indian politics. Your dedication to the cause of building @AamAadmiParty is an inspiration to us all. May God bless you with good health, long life and abundance of happiness. @AAPDelhi @AAPPunjab https://t.co/dXVvp7Y9GR",[],<NA>,<NA>,0
392,Blogging-- http://13tolife.us/ Mentioning links to contests where you might just win a free book or two!,[],4,<NA>,0
3975,"lol i know but it was just so funny, ahaha",[],4,<NA>,0
45648,@Awhadspeaks Kindly keep doing same social services and don't involve in politics. We need leader like Tata and you can follow same.,[],<NA>,<NA>,0
12424,Indian cricket team promised that they must defeat by any cricket team of world. The selection committee of BCCI now prove that they boast by politics. Nonsense Indian cricket team.,"[Indian, BCCI, Indian]",<NA>,<NA>,0
41201,hate u.... I have 2 wait one week to see it cuz here (Puerto Rico) is still coming soon....,[Puerto Rico],2,<NA>,0


In [ ]:
# ============================================================
# STEP: Combine labels using agreement-based voting
# (replaces the plain OR-based df['political_label'] assignment)
# ============================================================

# tweet_label = mention/hashtag family (high-confidence, from your verified top-300 dictionary)
# tfidf_label, ner_label, nmf_label, lda_label = noisier heuristic families

label_cols = ['tfidf_label', 'ner_label', 'nmf_label', 'lda_label']

# make sure any missing family columns default to 0 instead of breaking the sum
for col in label_cols:
    if col not in df.columns:
        df[col] = 0

df['family_agreement_count'] = df[label_cols].sum(axis=1)

MIN_FAMILIES_AGREEING = 2

df['political_label'] = 0

# tweet_label (mention/hashtag) is trusted on its own -> overrides everything
df.loc[df['tweet_label'] == 1, 'political_label'] = 1

# for tweets with NO mention/hashtag signal, require at least 2 of the
# noisier families (tfidf/ner/nmf/lda) to agree before calling it political
no_signal = df['tweet_label'] != 1
df.loc[no_signal & (df['family_agreement_count'] >= MIN_FAMILIES_AGREEING), 'political_label'] = 1

print(df['political_label'].value_counts())
print(f"Political: {df['political_label'].mean()*100:.2f}%")

print("\nSample POLITICAL:")
display(df[df['political_label'] == 1]['tweet'].sample(5, random_state=1))

print("\nSample NON-POLITICAL:")
display(df[df['political_label'] == 0]['tweet'].sample(5, random_state=1))

df.to_csv('final_political_labels.csv', index=False)

political_label
1    25378
0    22612
Name: count, dtype: int64
Political: 52.88%

Sample POLITICAL:


,tweet
37278,"@khanavo @Ajayvchandra1 @vikramchandra Modi looks (or is made to look) bigger than #BJP and BJP bigger than India! Cause for concern Isn’t it? By BJP’s own yard stick, #LKAdvani advice in priority_ Nation, Party and individuals."
28517,@HardikPatel_ @ahmedpatel @SATAVRAJEEV @avinashpandeinc @priyankagandhi @sherryontopp @RahulGandhi @ashokgehlot51 @JM_Scindia @AmitChavdaINC @NewIndianXpress #LokSabhaElections2019 What about patidar andalon???? People will not forget who had lost their life because of your political gain
23293,@sardesairajdeep @RahulGandhi @INCIndia how idiotically the #BJP in opposition is hell bent upon establishing one upmanship
27891,"@sagarikaghose what party politics? if after 7 decades of your akka rule, we did not have war memorial, its only #NarendraModi could do, what is harm speaking truth?\n\nwhen ur boss @RahulGandhi do same #KhoonKiDalali, why u silent?\n\n#IndiaStrikesBack #airstrike #Balakot"
41153,Yuzvendra Chahal gets his fourth wicket of the match. #INDvsENG 💓🔥🏏 #yuzvendrachahal #RohitSharma #JosButtler #moeenali #jonnybairstow #davidwilley #England #India #CricbuzzLive #CricketTwitter #Cricket #HardikPandya #MohammedShami #PlayBold #WhistlePodu #champion #BCCI #ICC https://t.co/0EaMoRTUZi



Sample NON-POLITICAL:


,tweet
9401,- whoa. 'sack chasing whores' ? that is soo out of character for you to say that... bring back my sweet Superman
23235,"NeeRain was awarded as an emerging startup in water conservation by EDII, Ahmedabad. Shri. Ashwini Vaishnav, Hon. Minister of Railways, Communications and Electronics &amp; Information Technology, Government of India. #rain #Water #governmentofindia #India #GroundWater #Neerain https://t.co/GVbazE0mGT"
34845,"mmm mmm mmm! tss tsss tss. LOL, having way too much fun being bored. i miss him"
10487,Past experiences of being a redhead (eg. discrimination) or even just general info bout gingers in society! It all helps
17625,"#HappyBirthday to my friend and colleague, Shri @raghav_chadha #youthicon of Indian politics. Your dedication to the cause of building @AamAadmiParty is an inspiration to us all. May God bless you with good health, long life and abundance of happiness. @AAPDelhi @AAPPunjab https://t.co/dXVvp7Y9GR"


In [ ]:
def apply_weighted_decision(row):
    """
    Priority-based decision making for tweet classification.

    Priority Order (Highest to Lowest):
    1. Mentions (political mentions from top-200)
    2. Hashtags (political hashtags from top-200)
    3. NER (Named Entity Recognition)
    4. NMF (Non-negative Matrix Factorization)
    5. LDA (Latent Dirichlet Allocation)

    Returns: (label, reason)
    """

    mentions = row.get('mention_filtered', [])
    hashtags = row.get('hashtags_filtered', [])
    ner_entities = row.get('ner_entities', [])
    nmf_dominant_topic = row.get('nmf_dominant_topic', pd.NA)
    lda_dominant_topic = row.get('lda_dominant_topic', pd.NA)

    # Priority 1: Mentions (highest priority)
    if mentions and len(mentions) > 0:
        is_political = bool(set(m.lower() for m in mentions) & political_mentions_set)
        label = 1 if is_political else 0
        reason = f"Mention-based: {mentions} -> {'political' if is_political else 'non-political'}"
        return label, reason

    # Priority 2: Hashtags
    if hashtags and len(hashtags) > 0:
        is_political = bool(set(h.lower() for h in hashtags) & political_hashtags_set)
        label = 1 if is_political else 0
        reason = f"Hashtag-based: {hashtags} -> {'political' if is_political else 'non-political'}"
        return label, reason

    # Priority 3: NER
    if ner_entities and len(ner_entities) > 0:
        is_political = any(
            e.lower().replace(" ", "") in (political_mentions_set | political_hashtags_set)
            for e in ner_entities
        )
        label = 1 if is_political else 0
        reason = f"NER-based: {ner_entities} -> {'political' if is_political else 'non-political'}"
        return label, reason

    # Priority 4: NMF
    if pd.notna(nmf_dominant_topic):
        reason = f"NMF-based: Topic {nmf_dominant_topic} -> political (topic overlap)"
        return 1, reason

    # Priority 5: LDA (lowest priority)
    if pd.notna(lda_dominant_topic):
        reason = f"LDA-based: Topic {lda_dominant_topic} -> political (topic overlap)"
        return 1, reason

    # Default: No clear signal
    return 0, "No clear signal - default Non-Political"


# Apply the priority-based decision
df[['priority_label', 'priority_reason']] = df.apply(
    lambda row: pd.Series(apply_weighted_decision(row)),
    axis=1
)
print(f"   Political tweets: {(df['priority_label'] == 1).sum():,}")
print(f"   Non-Political tweets: {(df['priority_label'] == 0).sum():,}")
print(f"   Total tweets: {len(df):,}")

print(f"\n📊 Decision Reasons Distribution:")
print(df['priority_reason'].value_counts())

print(f"\n{'='*100}")


PRIORITY-BASED HIERARCHICAL DECISION - FINAL RESULTS
   Political tweets: 10,332
   Non-Political tweets: 37,658
   Total tweets: 47,990

📊 Decision Reasons Distribution:
priority_reason
No clear signal - default Non-Political                                                             32742
LDA-based: Topic 2 -> political (topic overlap)                                                      4536
LDA-based: Topic 4 -> political (topic overlap)                                                      2255
LDA-based: Topic 0 -> political (topic overlap)                                                       782
LDA-based: Topic 3 -> political (topic overlap)                                                       661
                                                                                                    ...  
NER-based: ['pakistan', 'iafhaitomumkinhai', 'hindustankajawab jaihind'] -> political                   1
NER-based: ['Fuzzball', 'SNL'] -> non-political                       

In [ ]:


from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, confusion_matrix

baseline_model = LinearSVC(max_iter=5000)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)

print("=== Baseline SVM (default params) — Held-out Test Performance ===")
print(classification_report(y_test, y_pred_baseline, target_names=['Non-Political', 'Political']))

baseline_f1 = f1_score(y_test, y_pred_baseline)
print(f"Baseline Test F1: {baseline_f1:.4f}")

cm_baseline = confusion_matrix(y_test, y_pred_baseline)
print("\nConfusion Matrix (baseline):")
print(cm_baseline)

=== Baseline SVM (default params) — Held-out Test Performance ===
               precision    recall  f1-score   support

Non-Political       0.93      0.95      0.94      4522
    Political       0.96      0.94      0.95      5076

     accuracy                           0.95      9598
    macro avg       0.95      0.95      0.95      9598
 weighted avg       0.95      0.95      0.95      9598

Baseline Test F1: 0.9482

Confusion Matrix (baseline):
[[4317  205]
 [ 315 4761]]


In [ ]:


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import LinearSVC

# NOTE: only use clean_text/tweet + engineered non-leaky features as input.
# Do NOT include tweet_label, tfidf_label, ner_label, nmf_label, lda_label,
# or family_agreement_count as model features — those columns were used to
# CONSTRUCT political_label, so including them causes label leakage
# (perfect/tautological accuracy that means nothing).

X_text = df['clean_text']
y = df['political_label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

svm_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.9)
X_train = svm_vectorizer.fit_transform(X_train_text)
X_test = svm_vectorizer.transform(X_test_text)

print(f"Train shape: {X_train.shape}")
print(f"Test shape : {X_test.shape}")

param_grid = {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced']}
grid = GridSearchCV(LinearSVC(max_iter=5000), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV F1: {grid.best_score_:.4f}")

svm_model = grid.best_estimator_

Train shape: (38392, 5000)
Test shape : (9598, 5000)
Best params: {'C': 1, 'class_weight': None}
Best CV F1: 0.9482


In [ ]:
param_grid = {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced']}
grid = GridSearchCV(LinearSVC(max_iter=5000), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print(f"Best CV F1: {grid.best_score_:.4f}")

svm_model = grid.best_estimator_

Best params: {'C': 1, 'class_weight': None}
Best CV F1: 0.9482


In [ ]:
# ============================================================
# STEP: Tuned SVM — Held-out Test Evaluation
# ============================================================

from sklearn.metrics import classification_report, f1_score, confusion_matrix

y_pred_tuned = svm_model.predict(X_test)

print("=== Tuned SVM (GridSearchCV best params) — Held-out Test Performance ===")
print(classification_report(y_test, y_pred_tuned, target_names=['Non-Political', 'Political']))

tuned_f1 = f1_score(y_test, y_pred_tuned)
print(f"Tuned Test F1: {tuned_f1:.4f}")

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
print("\nConfusion Matrix (tuned):")
print(cm_tuned)

print(f"\nBaseline Test F1: {baseline_f1:.4f}")
print(f"Tuned Test F1   : {tuned_f1:.4f}")
print(f"Improvement     : {tuned_f1 - baseline_f1:+.4f}")

=== Tuned SVM (GridSearchCV best params) — Held-out Test Performance ===
               precision    recall  f1-score   support

Non-Political       0.93      0.95      0.94      4522
    Political       0.96      0.94      0.95      5076

     accuracy                           0.95      9598
    macro avg       0.95      0.95      0.95      9598
 weighted avg       0.95      0.95      0.95      9598

Tuned Test F1: 0.9482

Confusion Matrix (tuned):
[[4317  205]
 [ 315 4761]]

Baseline Test F1: 0.9482
Tuned Test F1   : 0.9482
Improvement     : +0.0000


In [ ]:
import joblib

joblib.dump(svm_model, 'political_classifier_svm.joblib')
joblib.dump(svm_vectorizer, 'political_tfidf_vectorizer.joblib')

print("Saved: political_classifier_svm.joblib")
print("Saved: political_tfidf_vectorizer.joblib")

Saved: political_classifier_svm.joblib
Saved: political_tfidf_vectorizer.joblib


In [ ]:
# ============================================================
# STEP: Test the trained model on brand-new, unseen tweets
# ============================================================

new_tweets = [
    "@narendramodi bjp government announces new policy for farmers welfare",
    "just watched a great movie with my friends tonight, loved it",
    "rahulgandhi congress slams government over inflation and unemployment",
    "happy birthday to my sister, wishing you all the best",
    "@amitshah addresses rally ahead of state elections",
    "@viratkohli scores century in today's match against australia",
    "excited to try out the new restaurant that opened near my house"
]

# IMPORTANT: use .transform(), never .fit_transform(), on new data —
# it must reuse the exact vocabulary/weights learned during training
X_new = svm_vectorizer.transform(new_tweets)

predictions = svm_model.predict(X_new)

results_df = pd.DataFrame({
    'tweet': new_tweets,
    'predicted_label': predictions,
    'prediction': ['Political' if p == 1 else 'Non-Political' for p in predictions]
})

display(results_df)


,tweet,predicted_label,prediction
0,@narendramodi bjp government announces new policy for farmers welfare,1,Political
1,"just watched a great movie with my friends tonight, loved it",0,Non-Political
2,rahulgandhi congress slams government over inflation and unemployment,1,Political
3,"happy birthday to my sister, wishing you all the best",0,Non-Political
4,@amitshah addresses rally ahead of state elections,1,Political
5,@viratkohli scores century in today's match against australia,0,Non-Political
6,excited to try out the new restaurant that opened near my house,0,Non-Political


In [ ]:
display(results_df)

In [ ]:
# ============================================================
# STEP: Final sample-tweet inference table
# tweet | mentions | hashtags | NER | NMF | LDA | predicted_label | predicted
# Each of mentions/hashtags/NER/NMF/LDA shows the actual matched
# words/topic extracted from that tweet — the label is derived from
# those extracted features (not the SVM's own TF-IDF vectorizer).
# ============================================================

import re
import numpy as np
import pandas as pd

new_tweets = [
    "@narendramodi bjp government announces new policy for farmers welfare",
    "just watched a great movie with my friends tonight, loved it",
    "rahulgandhi congress slams government over inflation and unemployment",
    "happy birthday to my sister, wishing you all the best",
    "@amitshah addresses rally ahead of state elections",
    "@viratkohli is the greatest cricketer in my opinion",
    "#BJP continues their crulety as farmer unions reject new bills",
    "excited to try out the new restaurant that opened near my house",
    "#india #politics india has seen many leaders but modiji is the best",
    "@farhan is best bollywood director in my opinion #movies #films"
]

political_mentions_set = set(mention_political_dict.keys())
political_hashtags_set = set(hashtag_political_dict.keys())

rows = []

for tweet in new_tweets:
    clean = preprocess(tweet)

    mentions_found = [m.lower() for m in re.findall(r'@(\w+)', tweet)]

    hashtags_found = [h.lower() for h in re.findall(r'#(\w+)', tweet)]

    doc = nlp(tweet)
    ner_entities = [ent.text for ent in doc.ents if ent.label_ in RELEVANT_ENTITY_LABELS]

    # --- NMF: top word(s) driving the assigned topic for this tweet ---
    nmf_words = []
    try:
        vec = count_vectorizer.transform([clean])
        topic_dist = nmf.transform(vec)[0]
        top_topic = int(np.argmax(topic_dist))
        feat_names = count_vectorizer.get_feature_names_out()
        top_term_idx = nmf.components_[top_topic].argsort()[::-1][:5]
        candidate_terms = [feat_names[i] for i in top_term_idx]
        # only keep terms that actually appear in this tweet's clean text
        nmf_words = [t for t in candidate_terms if t in clean]
    except Exception:
        pass

    # --- LDA: top word(s) driving the assigned topic for this tweet ---
    lda_words = []
    try:
        bow = dictionary.doc2bow(clean.split())
        topic_probs = lda_model.get_document_topics(bow)
        if topic_probs:
            top_topic = max(topic_probs, key=lambda x: x[1])[0]
            candidate_terms = [w for w, _ in lda_model.show_topic(top_topic, topn=10)]
            lda_words = [t for t in candidate_terms if t in clean]
    except Exception:
        pass

    # --- derive label from the extracted features (agreement vote) ---
    has_political_mention = bool(set(mentions_found) & political_mentions_set)
    has_political_hashtag = bool(set(hashtags_found) & political_hashtags_set)
    tweet_label = int(has_political_mention or has_political_hashtag)

    ner_label  = int(any(e.lower().replace(" ", "") in (political_mentions_set | political_hashtags_set) for e in ner_entities))
    nmf_label  = int(len(nmf_words) > 0)
    lda_label  = int(len(lda_words) > 0)

    family_agreement_count = ner_label + nmf_label + lda_label
    predicted_label = 1 if tweet_label == 1 else int(family_agreement_count >= 2)

    rows.append({
        'tweet': tweet,
        'mentions': mentions_found,
        'hashtags': hashtags_found,
        'NER': ner_entities,
        'NMF': nmf_words,
        'LDA': lda_words,
        'predicted_label': predicted_label,
        'predicted': 'Political' if predicted_label == 1 else 'Non-Political'
    })

final_inference_table = pd.DataFrame(rows)
display(final_inference_table)

,tweet,mentions,hashtags,NER,NMF,LDA,predicted_label,predicted
0,@narendramodi bjp government announces new policy for farmers welfare,[narendramodi],[],[],[bjp],"[modi, bjp]",1,Political
1,"just watched a great movie with my friends tonight, loved it",[],[],[],[],[love],0,Non-Political
2,rahulgandhi congress slams government over inflation and unemployment,[],[],[congress],[congress],[],1,Political
3,"happy birthday to my sister, wishing you all the best",[],[],[],"[day, happy]","[day, happy]",1,Political
4,@amitshah addresses rally ahead of state elections,[amitshah],[],[],[],[],1,Political
5,@viratkohli is the greatest cricketer in my opinion,[viratkohli],[],[],[],[],0,Non-Political
6,#BJP continues their crulety as farmer unions reject new bills,[],[bjp],[BJP],[bjp],[bjp],1,Political
7,excited to try out the new restaurant that opened near my house,[],[],[],[new],[],0,Non-Political
8,#india #politics india has seen many leaders but modiji is the best,[],"[india, politics]",[india],[politics],"[modi, india]",1,Political
9,@farhan is best bollywood director in my opinion #movies #films,[farhan],"[movies, films]",[@farhan],[],[],0,Non-Political
